In [158]:
# 데이터 파이프라인
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
import os
from dotenv import load_dotenv
from datetime import datetime

# 알고리즘
from sklearn.cluster import KMeans

# 시각화
import folium
import random

# 데이터 처리
import ast
import calendar
import numpy as np
from math import radians

In [159]:
class AutoContainerGeneration:
    def __init__(self, version=1, debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")

        load_dotenv()

    # 공통 MySQL 데이터 추출 메서드
    def fetch_data(self, query, ssh_host, ssh_user, ssh_private_key, mysql_host, mysql_port, mysql_user, mysql_password, mysql_database):
        """
        MySQL 데이터를 추출하여 DF로 변환
        """
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port)
            ) as tunnel:
                print("SSH 터널 연결 성공")

                with pymysql.connect(
                    host='127.0.0.1',
                    user=mysql_user,
                    passwd=mysql_password,
                    db=mysql_database,
                    charset='utf8',
                    port=tunnel.local_bind_port,
                    cursorclass=pymysql.cursors.DictCursor) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")

                        # 결과를 DF로 변환

                        df = pd.DataFrame(results)
                        return df

        except Exception as e:
            print(f"Error fetching data for: {e}")
            return None

    # 전체 데이터 추출
    def fetch_all_data(self):
        """
        MySQL 쿼리를 실행하여 Shipping Items와 Bunny Schedule 데이터를 DF로 변환
        """
        # SSH 및 MySQL 설정
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv(r"SSH_PRIVATE_KEY")
        

        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")

        # 우편번호 그룹 폴리곤 데이터 쿼리
        # 평일
        regular_zipcode_groups_polygon_query = """
        SELECT region,
        group_name,
        ST_AsText(geometry) AS geometry,
        zipcodes
        from zipcode_groups_polygon
        where weekday = 0;
        """
        # 주말
        weekend_zipcode_groups_polygon_query = """
        SELECT region,
        group_name,
        ST_AsText(geometry) AS geometry,
        zipcodes
        from zipcode_groups_polygon
        where weekday = 1;
        """

        # 각 쿼리 결과를 DataFrame으로 가져오기

        df_regular_zipcode = self.fetch_data(regular_zipcode_groups_polygon_query, ssh_host, ssh_user, ssh_private_key,
                                                  mysql_host, mysql_port, mysql_user, mysql_password, mysql_database)
        df_weekend_zipcode = self.fetch_data(weekend_zipcode_groups_polygon_query, ssh_host, ssh_user, ssh_private_key,
                                                  mysql_host, mysql_port, mysql_user, mysql_password, mysql_database)

        return df_regular_zipcode, df_weekend_zipcode

In [160]:
if __name__ == "__main__":
    generator = AutoContainerGeneration(version=2, debug=True)
    df_regular, df_weekend = generator.fetch_all_data()

    # 각 DataFrame 확인
    print("DF 생성 완료")

AutoContainerGeneration version: 2
SSH 터널 연결 성공
쿼리 실행 완료
SSH 터널 연결 성공
쿼리 실행 완료
DF 생성 완료


In [161]:
workflow_day_shipping_items_df = pd.read_csv('../git_csv/20250318_오버랩제외_전체물량.csv')

workflow_day_bunny_df = pd.read_csv('../git_csv/20250318_고정버니.csv')

In [ ]:
def group_and_map_zipcodes(zipcode_groups, shipping_csv_path):
    
    result_gdf = zipcode_groups

    # 우편번호-그룹 매핑 생성
    zipcode_to_group = {}
    for _, row in result_gdf.iterrows():
        group = row['group_name']
        # 문자열 형태의 zipcodes를 리스트로 변환
        zipcodes_group = ast.literal_eval(row['zipcodes'])
        for zipcode in zipcodes_group:
            zipcode_to_group[zipcode] = group

    # 배송 데이터 로드 및 그룹 매핑
    df_shipping = shipping_csv_path
    df_shipping['group'] = df_shipping['zipcode'].map(zipcode_to_group)
    df_shipping = df_shipping[~df_shipping['group'].isna()]
    
    return result_gdf, df_shipping

In [163]:
def visualize_clusters(updated_df):
    # 지도 초기화
    map_center = [updated_df['lat'].mean(), updated_df['lng'].mean()]  # 물품의 평균 좌표를 중심으로 설정
    delivery_map = folium.Map(location=map_center, zoom_start=12)
    
    # 'group' 컬럼의 고유한 값들을 가져오되, NaN은 제외
    unique_groups = updated_df['cluster_label'].dropna().unique()
    color_map = { group: f"#{random.randint(0, 0xFFFFFF):06x}" for group in unique_groups }
    
    # 기본 색상 설정 (NaN인 경우 사용할 색상)
    default_color = "#808080"

    # 동그라미 마커 추가
    for _, row in updated_df.iterrows():
        Area = row['Area']
        latitude = row['lat']
        longitude = row['lng']
        item_uuid = row['shipping_uuid']
        zipcode = row['zipcode']
        Type = row['driver_type']
        cluster_label = row['cluster_label']
        code = row['code']
        

        # group이 NaN인지 확인하여 기본 색상 지정
        if Type == 'BLUE':
            color = default_color
        else:
            color = color_map.get(cluster_label, default_color)
        

        tooltip = f"Area: {Area}<br>Shipping_uuid: {item_uuid}<br>lat: {latitude}<br>lng: {longitude}<br>Zipcode: {zipcode}<br>Type: {Type}<br>cluster_label: {cluster_label}<br>Sector_code: {code}"
        folium.CircleMarker(
            location=[latitude, longitude],
            radius=10,  # 동그라미 크기
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=tooltip
        ).add_to(delivery_map)

    return delivery_map

In [164]:
# ---------------------------------------------------------------------------
# 1) 그룹별 중심 좌표 재계산
# ---------------------------------------------------------------------------
def recalc_group_centroids(df, group_col='group', lat_col='lat', lng_col='lng'):
    """
    DataFrame 내 group별 위경도(lat, lng) 평균을 구하여 centroid(중심좌표) 정보를 dict 형태로 반환
    """
    centroids = df.groupby(group_col).apply(
        lambda sub_df: (sub_df[lat_col].mean(), sub_df[lng_col].mean())
    ).to_dict()
    return centroids

# 기존 중심점에서 데이터 이동 말고 가까운 데이터들 가져오기
def min_dist_to_group(row, group_points):
    # 유클리드 예시 (필요시 Haversine 대체)
    dists = ((group_points['lat'] - row['lat'])**2 
           + (group_points['lng'] - row['lng'])**2)
    return np.sqrt(dists.min())

In [165]:

# ---------------------------------------------------------------------------
# 2) 화이트(WHITE) 버니 클러스터링 로직
# 40이상인 그룹 클러스터를 통해 20이상 40미만으로 맞추기
# ---------------------------------------------------------------------------
def isolate_white_clusters(df, group_name):
    
    iteration = 1
    current_group = group_name
    group_stack = [group_name]  # 처리할 그룹을 스택에 저장
    
    def min_dist_to_small(row):
        # row(큰 클러스터 소속)의 lat, lng
        # 작은 클러스터의 모든 점과의 거리 계산 후 최솟값
        dists = ((small_cluster_df["lat"] - row["lat"])**2 
                    + (small_cluster_df["lng"] - row["lng"])**2)
        return np.sqrt(dists.min())
    
    def min_dist_to_large(row):
        dists = ((large_cluster_df["lat"] - row["lat"])**2 
                    + (large_cluster_df["lng"] - row["lng"])**2)
        return np.sqrt(dists.min())

    while group_stack:
        current_group = group_stack.pop()  # 스택에서 하나의 그룹을 꺼내 처리
        group_orders = df[df['group'] == current_group].copy()
        total_count = group_orders['shipping_uuid'].count()


        if total_count >= 60:
            print(f"{current_group}의 주문 수가 {total_count}건")
            

        # (1) 40건 미만이면 화이트 처리 중단
        if total_count < 40:
            print(f"그룹 {current_group}의 주문 수가 {total_count}건으로 40 미만이어서 추가 분리 종료")
            continue

        if 60 <= total_count <= 69:
            print(f"{total_count}건 이므로 한 클러스터 30으로 조정")
            group_orders['lat_rad'] = group_orders['lat'].apply(radians)
            group_orders['lng_rad'] = group_orders['lng'].apply(radians)
            coords = group_orders[['lat_rad', 'lng_rad']].to_numpy()
            km = KMeans(n_clusters=2, init='k-means++', random_state=42)
            labels = km.fit_predict(coords)
            group_orders['cluster'] = labels
            cluster_counts = group_orders['cluster'].value_counts()
            print(f"클러스터링 결과: {cluster_counts.to_dict()}")

            # 라벨별 주문 수 확인
            label_a = cluster_counts.index[0]
            count_a = cluster_counts.iloc[0]
            label_b = cluster_counts.index[1]
            count_b = cluster_counts.iloc[1]

            # 작은 클러스터/큰 클러스터 구분
            if count_a < count_b:
                label_small = label_a
                label_large = label_b
            elif count_a > count_b:
                label_small = label_b
                label_large = label_a
            else:
                # 두 클러스터 사이즈 동일 시 tie-break
                print("두 클러스터 사이즈 동일 -> 강제로 label_small=0, label_large=1")
                sorted_idx = sorted(cluster_counts.index)
                label_small = sorted_idx[0]
                label_large = sorted_idx[1]

            while True:
                
                # 반복적으로 작은 클러스터가 30건이 되도록 조정
                cluster_counts = group_orders['cluster'].value_counts()
                label_a = cluster_counts.index[0]
                count_a = cluster_counts.iloc[0]
                label_b = cluster_counts.index[1]
                count_b = cluster_counts.iloc[1]

                if count_a < count_b:
                    label_small = label_a
                    label_large = label_b
                elif count_a > count_b:
                    label_small = label_b
                    label_large = label_a
                else:
                    print("작은/큰 클러스터 사이즈 동일 -> tie-break")
                    sorted_idx = sorted(cluster_counts.index)
                    label_small = sorted_idx[0]
                    label_large = sorted_idx[1]

                count_small = cluster_counts[label_small]
                count_large = cluster_counts[label_large]

                if 30 <= count_a < 40 and 30 <= count_b < 40:
                    new_white_group_a = f"{current_group}_{label_a}_WHITE"
                    new_white_group_b = f"{current_group}_{label_b}_WHITE"

                    indices_white_a = group_orders[group_orders['cluster'] == label_a].index
                    indices_white_b = group_orders[group_orders['cluster'] == label_b].index

                    # 그룹명 변경
                    df.loc[indices_white_a, 'group'] = new_white_group_a
                    df.loc[indices_white_b, 'group'] = new_white_group_b
                    print(f"[화이트 분리 완료] 그룹 {new_white_group_a} {len(indices_white_a)}건, {new_white_group_b} {len(indices_white_b)}건 생성, driver_type='WHITE'로 업데이트")

                if count_small == 30:
                    # 정확히 30이면 반복 종료
                    break

                elif count_small < 30:
                    # 작은 클러스터가 부족분을 큰 클러스터에서 가져옴
                    deficit = 30 - count_small
                    available = count_large - 30
                    if available <= 0:
                        print(f"조정 불가: 큰 클러스터({label_large})에 여분 주문이 없어 이동 불가")
                        break
                    move_count = min(deficit, available)
                    print(f"작은 클러스터 부족: {deficit}건, donor 클러스터에서 {move_count}건 이동 시도")

                    donor_orders = group_orders[group_orders['cluster'] == label_large].copy()

                    # 각 행(row)에 대해, '작은 클러스터' 모든 점과의 거리 중 최소값 계산
                    small_cluster_df = group_orders[group_orders['cluster'] == label_small][["lat","lng"]].copy()

                    donor_orders['dist_to_small'] = donor_orders.apply(min_dist_to_small, axis=1)
                    donor_orders.sort_values('dist_to_small', ascending=True, inplace=True)
                    # 가장 가까운 순으로 move_count만큼 이동
                    orders_to_move = donor_orders.head(move_count)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_small

                else:  # count_small > 30
                    # 작은 클러스터가 초과분을 큰 클러스터로 이동
                    surplus = count_small - 30
                    print(f"작은 클러스터 초과: {surplus}건, 큰 클러스터로 이동 시도")

                    large_centroid = group_orders[group_orders['cluster'] == label_large][['lat', 'lng']].mean().values

                    small_orders = group_orders[group_orders['cluster'] == label_small].copy()

                    small_orders['dist_to_large'] = np.sqrt(
                        (small_orders['lat'] - large_centroid[0])**2 + (small_orders['lng'] - large_centroid[1])**2
                    )
                    # 원하는 순서(가장 가까운 데이터부터 이동 등) 정렬
                    small_orders.sort_values('dist_to_large', ascending=True, inplace=True)
                    orders_to_move = small_orders.head(surplus)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_large

                # 재계산
                cluster_counts = group_orders['cluster'].value_counts()
                print(f"이동 후 클러스터 분포: {cluster_counts.to_dict()}")

                if label_small not in cluster_counts:
                    # label_small 자체가 사라지는 경우(클러스터가 0건이 된 상황)
                    print("label_small 클러스터가 사라져서 종료")
                    break

                count_small = cluster_counts[label_small]
                if count_small == 30:
                    break

            # 작은 클러스터가 30건이면 화이트로 확정
            count_small = group_orders['cluster'].value_counts().get(label_small, 0)
            if count_small == 30:
                new_white_group = f"{current_group}_WHITE"
                # 작은 클러스터에 해당하는 주문 인덱스
                indices_white = group_orders[group_orders['cluster'] == label_small].index

                # 그룹명 변경
                df.loc[indices_white, 'group'] = new_white_group
                print(f"[화이트 분리 완료] 그룹 {new_white_group} (30건) 생성, driver_type='WHITE'로 업데이트")

                iteration += 1

                # 나머지(큰 클러스터) 처리
                remaining_indices = group_orders[group_orders['cluster'] == label_large].index
                remaining_count = df.loc[remaining_indices, 'shipping_uuid'].count()
                if remaining_count < 40:
                    print(f"남은 그룹의 주문 수가 {remaining_count}건으로 40 미만이어서 추가 분리 종료")
                    new_remaining_group = f"{current_group}_R"
                    df.loc[remaining_indices, 'group'] = new_remaining_group
                    print(f"남은 주문 {remaining_count}건은 그룹 {new_remaining_group}로 업데이트")
                    continue
    
        if 70 <= total_count <= 78:
            print(f"{total_count}건 이므로 큰 클러스터 39로 조정")

            group_orders['lat_rad'] = group_orders['lat'].apply(radians)
            group_orders['lng_rad'] = group_orders['lng'].apply(radians)
            coords = group_orders[['lat_rad', 'lng_rad']].to_numpy()
            km = KMeans(n_clusters=2, init='k-means++', random_state=42)
            labels = km.fit_predict(coords)
            group_orders['cluster'] = labels
            cluster_counts = group_orders['cluster'].value_counts()
            print(f"클러스터링 결과: {cluster_counts.to_dict()}")

            # 라벨별 주문 수 확인
            label_a = cluster_counts.index[0]
            count_a = cluster_counts.iloc[0]
            label_b = cluster_counts.index[1]
            count_b = cluster_counts.iloc[1]

            # 작은 클러스터/큰 클러스터 구분
            if count_a < count_b:
                label_small = label_a
                label_large = label_b
            elif count_a > count_b:
                label_small = label_b
                label_large = label_a
            else:
                # 두 클러스터 사이즈 동일 시 tie-break
                print("두 클러스터 사이즈 동일 -> 강제로 label_small=0, label_large=1")
                sorted_idx = sorted(cluster_counts.index)
                label_small = sorted_idx[0]
                label_large = sorted_idx[1]

            while True:
                
                # 반복적으로 작은 클러스터가 30건이 되도록 조정
                cluster_counts = group_orders['cluster'].value_counts()
                label_a = cluster_counts.index[0]
                count_a = cluster_counts.iloc[0]
                label_b = cluster_counts.index[1]
                count_b = cluster_counts.iloc[1]

                if count_a < count_b:
                    label_small = label_a
                    label_large = label_b
                elif count_a > count_b:
                    label_small = label_b
                    label_large = label_a
                else:
                    print("작은/큰 클러스터 사이즈 동일 -> tie-break")
                    sorted_idx = sorted(cluster_counts.index)
                    label_small = sorted_idx[0]
                    label_large = sorted_idx[1]

                count_small = cluster_counts[label_small]
                count_large = cluster_counts[label_large]

                if count_large == 39:
                    # 정확히 39이면 반복 종료
                    break

                elif count_large < 39:
                    # 큰 클러스터가 부족분을 작은 클러스터에서 가져옴
                    deficit = 39 - count_large
                    available = count_small - 30
                    if available <= 0:
                        print(f"조정 불가: 큰 클러스터({label_large})에 여분 주문이 없어 이동 불가")
                        break
                    move_count = min(deficit, available)
                    print(f"작은 클러스터 부족: {deficit}건, donor 클러스터에서 {move_count}건 이동 시도")
                    
                    donor_orders = group_orders[group_orders['cluster'] == label_small].copy()
                    large_cluster_df = group_orders[group_orders['cluster'] == label_large][["lat","lng"]].copy()

                    donor_orders['dist_to_large'] = donor_orders.apply(min_dist_to_large, axis=1)
                    donor_orders.sort_values('dist_to_large', ascending=True, inplace=True)
                    orders_to_move = donor_orders.head(move_count)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_large

                else:  # count_large > 39
                    # 큰 클러스터가 초과분을 작은 클러스터로 이동
                    surplus = count_large - 39
                    print(f"큰 클러스터 초과: {surplus}건, 작은 클러스터로 이동 시도")

                    small_centroid = group_orders[group_orders['cluster'] == label_small][['lat', 'lng']].mean().values

                    large_orders = group_orders[group_orders['cluster'] == label_large].copy()

                    large_orders['dist_to_small'] = np.sqrt(
                        (large_orders['lat'] - small_centroid[0])**2 + (large_orders['lng'] - small_centroid[1])**2
                    )
                    # 원하는 순서(가장 가까운 데이터부터 이동 등) 정렬
                    large_orders.sort_values('dist_to_small', ascending=True, inplace=True)
                    orders_to_move = large_orders.head(surplus)

                    if len(orders_to_move) == 0:
                        print("이동할 주문이 없어 추가 조정 불가능")
                        break

                    group_orders.loc[orders_to_move.index, 'cluster'] = label_small

                # 재계산
                cluster_counts = group_orders['cluster'].value_counts()
                print(f"이동 후 클러스터 분포: {cluster_counts.to_dict()}")

                if label_small not in cluster_counts:
                    # label_small 자체가 사라지는 경우(클러스터가 0건이 된 상황)
                    print("label_small 클러스터가 사라져서 종료")
                    break

                count_large = cluster_counts[label_large]
                if count_large == 39:
                    break

            # 큰 클러스터가 39건이면 화이트로 확정
            count_large = group_orders['cluster'].value_counts().get(label_large, 0)
            if count_large == 39:
                new_white_group = f"{current_group}_WHITE_{iteration}"
                # 큰 클러스터에 해당하는 주문 인덱스
                indices_white = group_orders[group_orders['cluster'] == label_large].index

                # 그룹명 변경
                df.loc[indices_white, 'group'] = new_white_group
                print(f"[화이트 분리 완료] 그룹 {new_white_group} (39건) 생성, driver_type='WHITE'로 업데이트")

                iteration += 1

                # 나머지(작은 클러스터) 처리
                remaining_indices = group_orders[group_orders['cluster'] == label_small].index
                remaining_count = df.loc[remaining_indices, 'shipping_uuid'].count()
                if remaining_count < 40:
                    print(f"남은 그룹의 주문 수가 {remaining_count}건으로 40 미만이어서 추가 분리 종료")
                    new_remaining_group = f"{current_group}_R_WHITE"
                    df.loc[remaining_indices, 'group'] = new_remaining_group
                    print(f"남은 주문 {remaining_count}건은 그룹 {new_remaining_group}로 업데이트")
                    continue

        print(f"\n[화이트 처리] 그룹 {current_group} 총 주문 수: {total_count}건. 클러스터링 시도...")

        # 2클러스터 KMeans (lat, lng 사용)
        group_orders['lat_rad'] = group_orders['lat'].apply(radians)
        group_orders['lng_rad'] = group_orders['lng'].apply(radians)
        coords = group_orders[['lat_rad', 'lng_rad']].to_numpy()
        km = KMeans(n_clusters=2, init='k-means++', random_state=42)
        labels = km.fit_predict(coords)
        group_orders['cluster'] = labels
        cluster_counts = group_orders['cluster'].value_counts()
        print(f"클러스터링 결과: {cluster_counts.to_dict()}")

        # 라벨별 주문 수 확인
        label_a = cluster_counts.index[0]
        count_a = cluster_counts.iloc[0]
        label_b = cluster_counts.index[1]
        count_b = cluster_counts.iloc[1]


        # 클러스터 후 두그룹의 물량이 40 이상일 때 다시 루프
        if count_a >= 40 and count_b >= 40:
            new_group_a = f"{current_group}_SPLIT_{label_a}"
            new_group_b = f"{current_group}_SPLIT_{label_b}"

            indices_a = group_orders[group_orders['cluster'] == label_a].index
            indices_b = group_orders[group_orders['cluster'] == label_b].index

            df.loc[indices_a, 'group'] = new_group_a
            df.loc[indices_b, 'group'] = new_group_b

            print(f"→ 두 클러스터가 모두 40건 이상. 새로운 그룹 {new_group_a}, {new_group_b} 생성. 다시 클러스터링 수행")
            
            # 두 개의 새로운 그룹을 스택에 추가하여 모두 처리하도록 함
            group_stack.append(new_group_a)
            group_stack.append(new_group_b)
            continue  # 다음 그룹을 처리하기 위해 루프 진행



        # 클러스터 결과 두 클러스터가 20이상 40미만일때 바로 화이트 배정
        if 20 <= count_a < 40 and 20 <= count_b < 40:
            new_white_group_a = f"{current_group}_WHITE_{label_a}"
            new_white_group_b = f"{current_group}_WHITE_{label_b}"

            # 20~39 조건에 만족하는 클러스터에 해당하는 주문 인덱스
            indices_white_1 = group_orders[group_orders['cluster'] == label_a].index
            indices_white_2 = group_orders[group_orders['cluster'] == label_b].index

            # 그룹명 변경
            df.loc[indices_white_1, 'group'] = new_white_group_a
            df.loc[indices_white_2, 'group'] = new_white_group_b

            print(f"클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 {new_white_group_a}, {new_white_group_b} driver_type='WHITE'로 업데이트")
            continue

        # 클러스터 결과 한개의 클러스터가 20이상 40미만이고 다른 클러스터가 40이상일 때 20이상 40미만의 클러스터 화이트 배정
        if 20 <= count_a < 40 and 40 <= count_b :
            new_white_group_a = f"{current_group}_WHITE_{label_a}"

             # 20~39 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
            indices_a = group_orders[group_orders['cluster'] == label_a].index
            indices_b = group_orders[group_orders['cluster'] == label_b].index

            # 그룹명 변경
            df.loc[indices_a, 'group'] = new_white_group_a
            
            df.loc[indices_b, 'group'] = f"{current_group}_R"
            print("A 화이트 배정")

            # current_group 을 해당 "_R" 으로 교체
            current_group = f"{current_group}_R"

            print(f"클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 {new_white_group_a} driver_type='WHITE'로 업데이트")

            # 40건 이상 남은 그룹을 다시 처리하도록 스택에 추가
            group_stack.append(current_group)

            continue
        else:
            pass

        if 20 <= count_b < 40 and 40 <= count_a:
            new_white_group_b = f"{current_group}_WHITE_{label_b}"

            # 20~39 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
            indices_a = group_orders[group_orders['cluster'] == label_a].index
            indices_b = group_orders[group_orders['cluster'] == label_b].index

            # 그룹명 변경
            df.loc[indices_b, 'group'] = new_white_group_b
            df.loc[indices_a, 'group'] = f"{current_group}_R"
            print("B 화이트 배정")

            # current_group 을 해당 "_R" 으로 교체
            current_group = f"{current_group}_R"

            print(f"클러스터 초기 그룹 물량 20건이상 40건 미만이므로 그룹 {new_white_group_b} driver_type='WHITE'로 업데이트")
            group_stack.append(current_group)

            continue
        else:
            pass

        

        # 작은 클러스터/큰 클러스터 구분
        if count_a < count_b:
            label_small = label_a
            label_large = label_b
        elif count_a > count_b:
            label_small = label_b
            label_large = label_a
        else:
            # 두 클러스터 사이즈 동일 시 tie-break
            print("두 클러스터 사이즈 동일 -> 강제로 label_small=0, label_large=1")
            sorted_idx = sorted(cluster_counts.index)
            label_small = sorted_idx[0]
            label_large = sorted_idx[1]

        while True:
            
            # 반복적으로 작은 클러스터가 20건이 되도록 조정
            cluster_counts = group_orders['cluster'].value_counts()
            label_a = cluster_counts.index[0]
            count_a = cluster_counts.iloc[0]
            label_b = cluster_counts.index[1]
            count_b = cluster_counts.iloc[1]

            if count_a < count_b:
                label_small = label_a
                label_large = label_b
            elif count_a > count_b:
                label_small = label_b
                label_large = label_a
            else:
                print("작은/큰 클러스터 사이즈 동일 -> tie-break")
                sorted_idx = sorted(cluster_counts.index)
                label_small = sorted_idx[0]
                label_large = sorted_idx[1]

            count_small = cluster_counts[label_small]
            count_large = cluster_counts[label_large]

            if count_small == 20:
                # 정확히 20이면 반복 종료
                break

            elif count_small < 20:
                # 작은 클러스터가 부족분을 큰 클러스터에서 가져옴
                deficit = 20 - count_small
                available = count_large - 20
                if available <= 0:
                    print(f"조정 불가: 큰 클러스터({label_large})에 여분 주문이 없어 이동 불가")
                    break
                move_count = min(deficit, available)
                print(f"작은 클러스터 부족: {deficit}건, donor 클러스터에서 {move_count}건 이동 시도")

                # 각 행(row)에 대해, '작은 클러스터' 모든 점과의 거리 중 최소값 계산
                donor_orders = group_orders[group_orders['cluster'] == label_large].copy()
                small_cluster_df = group_orders[group_orders['cluster'] == label_small][["lat","lng"]].copy()

                donor_orders['dist_to_small'] = donor_orders.apply(min_dist_to_small, axis=1)
                donor_orders.sort_values('dist_to_small', ascending=True, inplace=True)
                # 가장 가까운 순으로 move_count만큼 이동
                orders_to_move = donor_orders.head(move_count)

                if len(orders_to_move) == 0:
                    print("이동할 주문이 없어 추가 조정 불가능")
                    break

                group_orders.loc[orders_to_move.index, 'cluster'] = label_small

            else:
                # 작은 클러스터가 초과분을 큰 클러스터로 이동
                surplus = count_small - 20
                print(f"작은 클러스터 초과: {surplus}건, 큰 클러스터로 이동 시도")

                donor_orders = group_orders[group_orders['cluster'] == label_small].copy()
                large_cluster_df = group_orders[group_orders['cluster'] == label_large][["lat","lng"]].copy()

                donor_orders['dist_to_large'] = donor_orders.apply(min_dist_to_large, axis=1)
                donor_orders.sort_values('dist_to_large', ascending=True, inplace=True)
                orders_to_move = donor_orders.head(move_count)

                if len(orders_to_move) == 0:
                    print("이동할 주문이 없어 추가 조정 불가능")
                    break

                group_orders.loc[orders_to_move.index, 'cluster'] = label_large

            # 재계산
            cluster_counts = group_orders['cluster'].value_counts()
            print(f"이동 후 클러스터 분포: {cluster_counts.to_dict()}")

            if label_small not in cluster_counts:
                # label_small 자체가 사라지는 경우(클러스터가 0건이 된 상황)
                print("label_small 클러스터가 사라져서 종료")
                break

            count_small = cluster_counts[label_small]
            if count_small == 20:
                break

        # 작은 클러스터가 20건이면 화이트로 확정
        count_small = group_orders['cluster'].value_counts().get(label_small, 0)
        if count_small == 20:
            new_white_group = f"{current_group}_WHITE_{iteration}"
            # 작은 클러스터에 해당하는 주문 인덱스
            indices_white = group_orders[group_orders['cluster'] == label_small].index

            # 그룹명 변경
            df.loc[indices_white, 'group'] = new_white_group
            print(f"[화이트 분리 완료] 그룹 {new_white_group} (20건) 생성, driver_type='WHITE'로 업데이트")

            iteration += 1

            # 나머지(큰 클러스터) 처리
            remaining_indices = group_orders[group_orders['cluster'] == label_large].index
            remaining_count = df.loc[remaining_indices, 'shipping_uuid'].count()
            if remaining_count < 40:
                print(f"남은 그룹의 주문 수가 {remaining_count}건으로 40 미만이어서 추가 분리 종료")
                continue

            new_remaining_group = f"{current_group}_R"
            df.loc[remaining_indices, 'group'] = new_remaining_group
            print(f"남은 주문 {remaining_count}건은 그룹 {new_remaining_group}로 업데이트")

            # 스택에 새 그룹 push → 다음 while loop iteration에서 pop하여 재처리
            group_stack.append(new_remaining_group)
            
            continue
            

        else:
            print(f"화이트 분리 불가: 작은 클러스터가 20건이 아닙니다. (현재 {count_small}건)")
            continue

    

    return df

In [168]:

# ---------------------------------------------------------------------------
# 3) 고정버니
# 버니(YELLOW/RAINBOW/ORANGE) 배정 로직
# ---------------------------------------------------------------------------


def assign_fixed_drivers(df_shipping_region, fix_region_workflow_day_bunny_df):

    for col in ['driver_type', 'driver_code']:
        if col not in df_shipping_region.columns:
            df_shipping_region[col] = np.nan

    
    # 버니 우선순위 매핑 & 정렬
    type_priority = {'YELLOW': 1, 'RAINBOW': 2, 'ORANGE': 3}
    fix_region_workflow_day_bunny_df['type_prio'] = fix_region_workflow_day_bunny_df['Type'].map(type_priority)
    fix_region_workflow_day_bunny_df.sort_values(['type_prio'], inplace=True)

    # Y/R 버니
    #  ORANGE 버니로 분리
    non_orange_drivers = fix_region_workflow_day_bunny_df[fix_region_workflow_day_bunny_df['Type'] != 'ORANGE']
    orange_drivers = fix_region_workflow_day_bunny_df[fix_region_workflow_day_bunny_df['Type'] == 'ORANGE']


    # 버니 배정용 index
    assigned_non_orange = 0
    assigned_orange = 0

    # (A) YELLOW/RAINBOW 버니
    # 버니 배정: 40~50건 그룹
    group_counts = df_shipping_region.groupby('group')['shipping_uuid'].count()
    yellow_rainbow_groups = group_counts[(group_counts >= 40) & (group_counts < 50)].index
    print(f"[Y/R 버니 수] = {len(non_orange_drivers)}")
    print(f"[ORANGE 버니 수] = {len(orange_drivers)}")

    for grp in yellow_rainbow_groups:
        if assigned_non_orange >= len(non_orange_drivers):
            break
        grp_orders = df_shipping_region[df_shipping_region['group'] == grp]
        new_grp_label = f"{grp}_Y/R"
        selected_idx = grp_orders.index
        df_shipping_region.loc[selected_idx, 'group'] = new_grp_label

        driver = non_orange_drivers.iloc[assigned_non_orange]
        
        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_type'] = driver['Type']
        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_code'] = driver['code']

        assigned_non_orange += 1
        print(f"{grp}에 {driver['Type']} 배정")
        # 최신 그룹 상태 반영
        group_centroids = recalc_group_centroids(df_shipping_region)

    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
    print(f"[Y/R 1차 배정] 40~50건 그룹 배정 후 남은 Yellow/Rainbow 버니 = {leftover_non_orange}")

    # (B) ORANGE 버니
    #  배정: 20~29건 그룹
    group_counts = df_shipping_region.groupby('group')['shipping_uuid'].count()
    orange_groups = group_counts[(group_counts >= 20) & (group_counts <= 29)].index

    for grp in orange_groups:
        if assigned_orange >= len(orange_drivers):
            break
        grp_orders = df_shipping_region[df_shipping_region['group'] == grp]
        new_grp_label = f"{grp}_O"
        selected_idx = grp_orders.index
        df_shipping_region.loc[selected_idx, 'group'] = new_grp_label

        driver = orange_drivers.iloc[assigned_orange]
        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_type'] = driver['Type']
        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_code'] = driver['code']
        assigned_orange += 1
        print(f"{grp}에 {driver['Type']} 배정")
        # 최신 그룹 상태 반영
        group_centroids = recalc_group_centroids(df_shipping_region)

    leftover_orange = len(orange_drivers) - assigned_orange
    print(f"[O 1차 배정] 오렌지 버니 배정 후 남은 버니 = {leftover_orange}")

    # ------------------------------------------------------------------
    # 아래는 추가 재배정 로직(주문수 50~60 이상인 그룹, 40~55 그룹, etc.)
    # 기존 코드를 그대로 옮겨와 구조만 함수 안에 배치한 것
    # ------------------------------------------------------------------

    # 1) Y/R 2차 재배정 (주문수 >= 60 그룹)
    yr_group_counter = {}
    stop_all = False

    print("### [Y/R 2차 재배정] ###")
    for _ in range(len(non_orange_drivers)):
        if leftover_non_orange <= 0:
            stop_all = True
            break

        unassigned_df = df_shipping_region[df_shipping_region['driver_type'].isna()]

        group_counts = unassigned_df.groupby('group')['shipping_uuid'].count()
        extra_unassigned_max_groups = group_counts[group_counts >= 60].index.tolist()
        print(f"배정되지 않은 그룹 {extra_unassigned_max_groups}")
        print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
        
        if not extra_unassigned_max_groups:
            break

                    
        for grp in extra_unassigned_max_groups:
            if leftover_non_orange <= 0:
                stop_all = True
                break
        
            group_stack = [grp]  
            
            while group_stack and not stop_all:
                current_group = group_stack.pop()  # 스택에서 하나의 그룹을 꺼내 처리
                grp_orders = df_shipping_region[df_shipping_region['group'] == current_group].copy()

                if grp_orders.shape[0] < 60:
                    continue
                current_count = df_shipping_region[df_shipping_region['group'] == current_group]['shipping_uuid'].count()
                print(f"[Y/R 2차 재배정] 그룹 [{current_group}] (주문수: {current_count})에서 클러스터 추출 시도")

                grp_orders_rad = grp_orders.copy()
                grp_orders_rad['lat_rad'] = grp_orders_rad['lat'].apply(radians)
                grp_orders_rad['lng_rad'] = grp_orders_rad['lng'].apply(radians)
                coords = grp_orders_rad[['lat_rad', 'lng_rad']].to_numpy()
                km = KMeans(n_clusters=2, init='k-means++', random_state=42)
                cluster_labels = km.fit_predict(coords)
                grp_orders_rad['cluster'] = cluster_labels
                cluster_counts = grp_orders_rad['cluster'].value_counts()
                if len(cluster_counts) < 2:
                    continue
                
                # 라벨별 주문 수 확인
                label_a = cluster_counts.index[0]
                count_a = cluster_counts.iloc[0]
                label_b = cluster_counts.index[1]
                count_b = cluster_counts.iloc[1]

                # 작은/큰 클러스터 분할
                if cluster_counts.iloc[0] <= cluster_counts.iloc[1]:
                    smaller_cluster = cluster_counts.index[0]
                    larger_cluster = cluster_counts.index[1]
                else:
                    smaller_cluster = cluster_counts.index[1]
                    larger_cluster = cluster_counts.index[0]

                print(f"  → 그룹 [{grp}] 클러스터 결과: 클러스터1 {label_a} ({cluster_counts[label_a]}건), "
                        f"larger 클러스터 {label_b} ({cluster_counts[label_b]}건)")
                
                        
                if 55 < count_a < 60 and count_b >= 60:
                    # a는 종료
                    new_group_a = f"{current_group}_R_{label_a}"
                    # b는 스택으로 넘기고
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    print(f"클러스터 {label_a}, {label_b}")

                    # 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index
            
                    # 그룹명 변경
                    df_shipping_region.loc[indices_a, 'group'] = new_group_a
                    df_shipping_region.loc[indices_b, 'group'] = new_group_b

                    print(f"→ 클러스터{label_a}: remain, 클러스터{label_b}: stack 넘겨 다시 클러스터링 수행")
                    
                    group_stack.append(new_group_b)
                    continue  # 다음 그룹을 처리하기 위해 루프 진행

                if 55 < count_b < 60 and count_a >= 60:
                    # b는 종료
                    new_group_b = f"{current_group}_R_{label_b}"
                    # a는 스택으로 넘기고
                    new_group_a = f"{current_group}_SPLIT_{label_a}"

                    print(f"클러스터 {label_a}, {label_b}")

                    # 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index
            
                    # 그룹명 변경
                    df_shipping_region.loc[indices_a, 'group'] = new_group_a
                    df_shipping_region.loc[indices_b, 'group'] = new_group_b

                    print(f"→ 클러스터{label_b}: remain, 클러스터{label_a}: stack 넘겨 다시 클러스터링 수행")
                    
                    group_stack.append(new_group_a)
                    continue  # 다음 그룹을 처리하기 위해 루프 진행

                if count_a >= 60 and count_b >= 60:
                    new_group_a = f"{current_group}_SPLIT_{label_a}"
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index

                    df_shipping_region.loc[indices_a, 'group'] = new_group_a
                    df_shipping_region.loc[indices_b, 'group'] = new_group_b

                    print(f"→ 두 클러스터가 모두 60건 이상. 새로운 그룹 {new_group_a}, {new_group_b} 생성. 다시 클러스터링 수행")
                    
                    # 두 개의 새로운 그룹을 스택에 추가하여 모두 처리하도록 함
                    group_stack.append(new_group_a)
                    group_stack.append(new_group_b)
                    continue  # 다음 그룹을 처리하기 위해 루프 진행

                if 40 <= count_a <= 55 and count_b >= 60:
                    new_YR_group_a = f"{current_group}_Y/R_{label_a}"
                    # 큰 클러스터
                    new_big_grp = f"{grp}_remain"

                    new_counts = grp_orders_rad['cluster'].value_counts()
                    new_count = new_counts.get(label_a, 0)
                    print(f"클러스터 {label_a} 주문수: {new_count} (목표:40 ~ 55)")

                    # 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index
            
                    # 그룹명 변경
                    df_shipping_region.loc[indices_a, 'group'] = new_YR_group_a
                    df_shipping_region.loc[indices_b, 'group'] = new_big_grp

                    if assigned_non_orange < len(non_orange_drivers):
                        driver = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_a, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_a, 'driver_code'] = driver['code']

                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_a}] (약 {new_count}건) → [{driver['Type']}] 배정")
                        
                        assigned_non_orange += 1
                        leftover_non_orange -= 1
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break
                    

                    print(f"클러스터 초기 그룹 물량 40건이상 55건 이하 이므로 그룹 {new_YR_group_a} driver_type={driver['Type']}에게 배정")

                    # 60건 이상 남은 그룹을 다시 처리하도록 스택에 추가
                    group_stack.append(new_big_grp)

                    continue

                if 40 <= count_b <= 55 and count_a >= 60:
                    new_YR_group_b = f"{current_group}_Y/R_{label_b}"
                    # 큰 클러스터
                    new_big_grp = f"{grp}_remain"
                    
                    new_counts = grp_orders_rad['cluster'].value_counts()
                    new_count = new_counts.get(label_b, 0)
                    print(f"클러스터 {label_b} 주문수: {new_count} (목표:40 ~ 55)")

                    # 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index
                    
                    
                    # 그룹명 변경
                    df_shipping_region.loc[indices_b, 'group'] = new_YR_group_b
                    df_shipping_region.loc[indices_a, 'group'] = new_big_grp
                    
                    if assigned_non_orange < len(non_orange_drivers):
                        driver = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_b, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_b, 'driver_code'] = driver['code']

                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_b}] (약 {new_count}건) → [{driver['Type']}] 배정")
                        
                        assigned_non_orange += 1
                        leftover_non_orange -= 1
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break
                    
                    print(f"클러스터 초기 그룹 물량 40건이상 55건 이하 이므로 그룹 {new_YR_group_b} driver_type={driver['Type']}에게 배정")

                    # 60건 이상 남은 그룹을 다시 처리하도록 스택에 추가
                    group_stack.append(new_big_grp)

                    continue

                if 40 <= count_a <= 55 and 40 <= count_b <= 55:
                    new_YR_group_a = f"{current_group}_Y/R_{label_a}"
                    new_YR_group_b = f"{current_group}_Y/R_{label_b}"

                    print(f"클러스터 {label_a}, {label_b} 주문수: (목표:40 ~ 55)")
                    
                    new_counts = grp_orders_rad['cluster'].value_counts()
                    new_count_a = new_counts.get(label_a, 0)
                    new_count_b = new_counts.get(label_b, 0)

                    # 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index
            
                    # 그룹명 변경
                    df_shipping_region.loc[indices_a, 'group'] = new_YR_group_a
                    df_shipping_region.loc[indices_b, 'group'] = new_YR_group_b

                    if assigned_non_orange < len(non_orange_drivers):
                        driver = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_a, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_a, 'driver_code'] = driver['code']

                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_a}] (약 {new_count_a}건) {driver['Type']} 배정")
                        
                        assigned_non_orange += 1
                        leftover_non_orange -= 1
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    if assigned_non_orange < len(non_orange_drivers):
                        driver = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_b, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_b, 'driver_code'] = driver['code']

                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_b}] (약 {new_count_b}건) {driver['Type']} 배정")
                        
                        assigned_non_orange += 1
                        leftover_non_orange -= 1
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break
                    
                    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                    print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
                    continue


                if 40 <= count_a <= 55 and 20 <= count_b < 40:
                    new_YR_group_a = f"{current_group}_Y/R_{label_a}"
                    new_group_b = f"{current_group}_R_{label_b}"
                    print(f"클러스터 {label_a}, {label_b}")

                    # 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index
            
                    # 그룹명 변경
                    df_shipping_region.loc[indices_a, 'group'] = new_YR_group_a
                    df_shipping_region.loc[indices_b, 'group'] = new_group_b

                    if assigned_non_orange < len(non_orange_drivers):
                        driver = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_a, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_a, 'driver_code'] = driver['code']
                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_a}] (약 {len(indices_a)}건) {driver['Type']} 배정")
                        
                        assigned_non_orange += 1
                        leftover_non_orange -= 1
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                    print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
                    continue

                
                if 40 <= count_b <= 55 and 20 <= count_a < 40:
                    new_YR_group_b = f"{current_group}_Y/R_{label_b}"
                    new_group_a = f"{current_group}_R_{label_a}"
                    print(f"클러스터 {label_a}, {label_b}")

                    # 조건에 만족하는 a 클러스터에 해당하는 주문 인덱스
                    indices_a = grp_orders_rad[grp_orders_rad['cluster'] == label_a].index
                    indices_b = grp_orders_rad[grp_orders_rad['cluster'] == label_b].index
            
                    # 그룹명 변경
                    df_shipping_region.loc[indices_a, 'group'] = new_group_a
                    df_shipping_region.loc[indices_b, 'group'] = new_YR_group_b

                    if assigned_non_orange < len(non_orange_drivers):
                        driver = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_b, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_YR_group_b, 'driver_code'] = driver['code']

                        print(f"[Y/R 2차 재배정] 그룹[{new_YR_group_b}] (약 {len(indices_a)}건) → [{driver['Type']}] 배정")
                        
                        assigned_non_orange += 1
                        leftover_non_orange -= 1
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break

                    leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                    print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")
                    continue
                
                if leftover_non_orange <= 0:
                    print("더 이상 Y/R 버니가 없습니다.")
                    stop_all = True
                    break


                current = cluster_counts[larger_cluster]
                if current < 40:
                    deficit = 40 - current
                    print(f"  → 부족: {deficit}건 필요")
                    smaller_orders = grp_orders_rad[grp_orders_rad['cluster'] == smaller_cluster].copy()

                    # 상대 클러스터(목적지) = larger_cluster -> 그 모든 (lat,lng)
                    larger_points = grp_orders_rad[grp_orders_rad['cluster'] == larger_cluster][['lat','lng']]

                    # 각 주문(row)에 대해 'larger_points'와의 최솟값을 계산
                    smaller_orders['dist_to_larger'] = smaller_orders.apply(
                        lambda row: min_dist_to_group(row, larger_points),
                        axis=1
                    )

                    # 가까운 순으로 정렬
                    smaller_orders.sort_values('dist_to_larger', inplace=True)
                    # deficit 건만 이동
                    indices_to_move = smaller_orders.index[:deficit]
                    
                    print(f"  → {deficit}건을 smaller 클러스터에서 larger 클러스터로 이동")
                    

                    # 버니 정보 초기화
                    df_shipping_region.loc[indices_to_move, ['driver_type', 'driver_code']] = np.nan
                    grp_orders_rad.loc[indices_to_move, 'cluster'] = larger_cluster
                
                elif 40 < current:
                    surplus = current - 40
                    print(f"  → 초과: {surplus}건 제거 필요")

                    larger_orders = grp_orders_rad[grp_orders_rad['cluster'] == larger_cluster].copy()

                    # 상대 클러스터(목적지) = smaller_cluster -> 그 모든 (lat,lng)
                    smaller_points = grp_orders_rad[grp_orders_rad['cluster'] == smaller_cluster][['lat','lng']]

                    # 각 주문(row)에 대해 'smaller_points'와의 최솟값 계산
                    larger_orders['dist_to_small'] = larger_orders.apply(
                        lambda row: min_dist_to_group(row, smaller_points),
                        axis=1
                    )
                    # 가까운 순으로 정렬
                    larger_orders.sort_values('dist_to_small', inplace=True)
                    # surplus 건만 이동
                    indices_to_move = larger_orders.index[:surplus]

                    
                    print(f"  → {surplus}건을 큰 클러스터에서 작은 클러스터로 이동")
                    df_shipping_region.loc[indices_to_move, ['driver_type', 'driver_code']] = np.nan
                    grp_orders_rad.loc[indices_to_move, 'cluster'] = smaller_cluster

                new_counts = grp_orders_rad['cluster'].value_counts()
                new_larger_count = new_counts.get(larger_cluster, 0)
                print(f"  → 조정 후: 큰 클러스터 {larger_cluster} 주문수: {new_larger_count} (목표:40)")

                if new_larger_count == 40:
                    base_name = grp.split('_cluster')[0]
                    yr_group_counter.setdefault(base_name, 0)
                    yr_group_counter[base_name] += 1
                    new_grp_label_large = f"{base_name}_cluster_Y/R_{yr_group_counter[base_name]}"
                    new_grp_label_small = f"{grp}_remaining"
                    indices_large = grp_orders_rad[grp_orders_rad['cluster'] == larger_cluster].index
                    indices_small = grp_orders_rad[grp_orders_rad['cluster'] == smaller_cluster].index
                    df_shipping_region.loc[indices_large, 'group'] = new_grp_label_large
                    df_shipping_region.loc[indices_small, 'group'] = new_grp_label_small
                    if assigned_non_orange < len(non_orange_drivers):
                        driver = non_orange_drivers.iloc[assigned_non_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label_large, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label_large, 'driver_code'] = driver['code']

                        print(f"[Y/R 2차 재배정] 그룹[{new_grp_label_large}] (약 {new_larger_count}건) → [{driver['Type']}] 배정")
                        
                        assigned_non_orange += 1
                        leftover_non_orange -= 1
                        group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print("[Y/R 2차 재배정] 배정 가능한 Y/R 버니가 더 이상 없습니다.")
                        break
                print("[Y/R 2차 재배정] 후 그룹별 물량:\n", df_shipping_region.groupby('group')['shipping_uuid'].count())

                if leftover_non_orange <= 0:
                    break

            if stop_all:
                break

        leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
        print(f"[Y/R 2차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")

        if stop_all:
            break
        
    print("[Y/R 2차 재배정] 로직 완료 or 버니 부족")

    # 2) Y/R 3차 재배정 40~55건 그룹에 남은 Y/R 버니 배정
    print("### [Y/R 3차 재배정] ###")
    for _ in range(len(non_orange_drivers)):
        if leftover_non_orange <= 0:
            print("남은 버니 부족으로 종료")
            break
        unassigned_df = df_shipping_region[df_shipping_region['driver_type'].isna()]
        unassigned_group_counts = unassigned_df.groupby('group')['shipping_uuid'].count()
        y_r_group = unassigned_group_counts[(unassigned_group_counts >= 40) & (unassigned_group_counts <= 55)].index.tolist()
        print("배정되지 않은 그룹: ", y_r_group)

        for grp in y_r_group:
            if leftover_non_orange <= 0:
                print("[Y/R 3차 재배정] 남은 40~55건의 그룹이 없음")
                break
            new_grp_label = f"{grp}_Y/R"
            grp_orders = df_shipping_region[df_shipping_region['group'] == grp]
            selected_idx = grp_orders.index
            df_shipping_region.loc[selected_idx, 'group'] = new_grp_label
            driver = non_orange_drivers.iloc[assigned_non_orange]
            df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_type'] = driver['Type']
            df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_code'] = driver['code']

            print(f"[Y/R 3차 재배정] 그룹[{new_grp_label}] (물량={group_counts[grp]}) → Y/R 버니[{driver['uuid']}] 배정")
            
            assigned_non_orange += 1
            leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
            group_centroids = recalc_group_centroids(df_shipping_region)
            print("[Y/R 3차 재배정] 후 그룹별 물량:\n", df_shipping_region.groupby('group')['shipping_uuid'].count())
        
        leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
        print(f"[Y/R 3차 재배정] 남은 Y/R 버니: {leftover_non_orange}명")

    
    # 3) 나머지 미배정 그룹들 중 큰 그룹 -> Y/R로 추가 배정
    # ---------------------------
    print("### [Y/R 4차 재배정] ###")
    for _ in range(len(non_orange_drivers)):
        if leftover_non_orange <= 0:
            print("남은 버니 부족으로 종료")
            break
        unassigned_groups = df_shipping_region[df_shipping_region['driver_type'].isna()]['group'].unique()

        if len(unassigned_groups) == 0:
            print("[Y/R 4차 재배정] 배정되지 않은 그룹이 없습니다.")
            break
        unassigned_group_counts = df_shipping_region[df_shipping_region['group'].isin(unassigned_groups)] \
                                    .groupby('group')['shipping_uuid'].count()
        
        unassigned_max_group = unassigned_group_counts.idxmax()
        print(f"선택된 unassigned_max_group (물량 많은 그룹): {unassigned_max_group} (물량: {unassigned_group_counts[unassigned_max_group]}건)")

        if unassigned_max_group in group_centroids:
            unassigned_max_group_centroid = group_centroids[unassigned_max_group]
        else:
            unassigned_group_orders = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]
            unassigned_max_group_centroid = (unassigned_group_orders['lat'].mean(), unassigned_group_orders['lng'].mean())
            group_centroids[unassigned_max_group] = unassigned_max_group_centroid

        unassigned_group_orders = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]
        plus_candidate_groups = [g for g in group_centroids.keys() if g != unassigned_max_group]
        
            
        
        if plus_candidate_groups:
            distances = {}
            for g in plus_candidate_groups:
                centroid = group_centroids[g]
                dist = np.sqrt((unassigned_max_group_centroid[0] - centroid[0])**2 + (unassigned_max_group_centroid[1] - centroid[1])**2)
                distances[g] = dist
            nearest_group = min(distances, key=distances.get)
            print(f"unassigned_max_group 과 가장 가까운 그룹: {nearest_group}")

            nearest_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
            
            if not nearest_orders.empty:
                nearest_driver_type = nearest_orders['driver_type'].iloc[0]
            else:
                nearest_driver_type = None

            print(f"근접 그룹 {nearest_group}의 driver_type: {nearest_driver_type}")

            if nearest_group in group_centroids:
                nearest_group_centroid = group_centroids[nearest_group]
            else:
                nearest_group_centroid = (nearest_orders['lat'].mean(), nearest_orders['lng'].mean())
                group_centroids[nearest_group] = nearest_group_centroid

            volume = unassigned_group_counts[unassigned_max_group]
            nearest_volume = nearest_orders['shipping_uuid'].count()
            
            if pd.isna(nearest_driver_type):
                print(f"근접 그룹 {nearest_group}의 driver_type이 null이므로, unassigned_max_group 의 물량이 40이 될 때까지 데이터를 이동합니다.")

                if volume < 40:
                    temp = nearest_orders.copy()
                    unassigned_points = unassigned_group_orders[['lat','lng']]
                    temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, unassigned_points), axis=1)
                    
                    temp.sort_values('dist_to_A', inplace=True, ascending=True)
                    needed = 40 - volume
                    available = nearest_volume - 20  # 이동 가능한 최대 주문 수
                    move_count = min(needed, available)
                    orders_to_move = temp.head(move_count)
                    print(f"[Y/R 4차 재배정] unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                    if  needed > available:
                        print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                        break


                    # 이미 배정된 버니정보가 있을 수도 있으므로 초기화
                    df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                    df_shipping_region.loc[orders_to_move.index, 'group'] = unassigned_max_group
                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                    if unassigned_max_group_volume >= 40:
                        if assigned_non_orange < len(non_orange_drivers):
                            driver = non_orange_drivers.iloc[assigned_non_orange]
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                            new_name = f"{unassigned_max_group}_Y/R"
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                            print(f"[Y/R 4차 재배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                            assigned_non_orange += 1
                            leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                            group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                        break


                elif volume >= 50:
                    temp = unassigned_group_orders.copy()
                    nearest_group_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]
                    temp['dist_to_A'] = temp.apply(lambda row: min_dist_to_group(row, nearest_group_points), axis=1)

                    temp.sort_values('dist_to_A', inplace=True)
                    needed = volume - 49
                    orders_to_move = temp.head(needed)
                    print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                    if len(orders_to_move) == 0:
                        print("이동할 데이터가 없으므로 종료")
                        break

                    # 이미 배정된 버니 정보가 있을 수도 있으므로 초기화
                    df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                    # nearest_group으로 지정
                    df_shipping_region.loc[orders_to_move.index, 'group'] = nearest_group
                    
                    
                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                    
                    # unassigned_max_group_volume이 49가 되면 버니 배정
                    if unassigned_max_group_volume <= 49:
                        if assigned_non_orange < len(non_orange_drivers):
                            driver = non_orange_drivers.iloc[assigned_non_orange]
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                            new_name = f"{unassigned_max_group}_Y/R"
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                            print(f"[Y/R 4차 재배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                            assigned_non_orange += 1
                            leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                            group_centroids = recalc_group_centroids(df_shipping_region)
               

            elif nearest_driver_type in ['YELLOW', 'RAINBOW']:
                # 근접 그룹(Y/R 그룹)의 물량이 40 미만이면 안된다.
                if nearest_volume < 40:
                    print(f"근접 그룹 {nearest_group}의 물량이 40 미만({nearest_volume}건)이라 unassigned_max_group 에 데이터를 가져올 수 없습니다. 종료합니다.")

                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 Y/R입니다. unassigned_max_group 의 물량이 40이 될 때까지 데이터 이동 시도합니다.")

                    if volume < 40:
                        temp = nearest_orders.copy()
                        unassigned_points = unassigned_group_orders[['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, unassigned_points), axis=1)

                        
                        temp.sort_values('dist_to_A', inplace=True)
                        needed = 40 - volume
                        available = nearest_volume - 40  # 이동 가능한 최대 주문 수
                        move_count = min(needed, available)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                            break

                        # 이동하는 주문들의 driver 관련 컬럼을 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                        df_shipping_region.loc[orders_to_move.index, 'group'] = unassigned_max_group

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        if unassigned_max_group_volume >= 40:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                print(f"[Y/R 4차 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                                
                                assigned_non_orange += 1
                                leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break
                    
                    # nearest_group에게 넘기기
                    elif volume >= 56:
                        if nearest_group in group_centroids:
                            nearest_group_centroid = group_centroids[nearest_group]
                        else:
                            nearest_group_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
                            nearest_group_centroid = (nearest_group_orders['lat'].mean(), nearest_group_orders['lng'].mean())
                            group_centroids[nearest_group] = nearest_group_centroid

                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()
                        
                        nearest_group_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda row: min_dist_to_group(row, nearest_group_points), axis=1)
                        
                        temp.sort_values('dist_to_A', inplace=True)
                        # 60이상은 위에서 이미 클러스터링 돼서 남아있는 그룹은 50~59
                        needed = volume - 55
                        available = 55 - nearest_volume  # 이동 가능한 최대 주문 수 => 배정이 안 될 수도 있으므로, 여유 둠.(최후)
                        move_count = min(needed, available)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                            break


                        # (1) 이동하는 주문들의 driver 컬럼 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                        # (2) group을 nearest_group으로 변경
                        df_shipping_region.loc[orders_to_move.index, 'group'] = nearest_group

                         # (3) 만약 nearest_group에 (Y/R 등)가 이미 배정되어 있다면, 그 버니정보를 이동된 주문에도 적용
                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group)
                            & (df_shipping_region['driver_type'].notna())
                        ].head(1)

                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]

                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code

                        # 이후 물량 카운트 버니 배정 처리
                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")


                        if unassigned_max_group_volume <= 55:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']
                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                print(f"[4차 Y/R 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")

                                assigned_non_orange += 1
                                leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break


            elif nearest_driver_type == 'ORANGE':
                # 근접 그룹(Orange 그룹)의 물량이 20 미만이면 안된다.
                if nearest_volume < 20:
                    
                    print(f"근접 그룹 {nearest_group}의 물량이 20 미만({nearest_volume}건)이라 unassigned_max_group 에 데이터를 가져올 수 없습니다. 종료합니다.")
                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 Orange입니다. unassigned_max_group 의 물량이 20이 될 때까지 데이터 이동 시도합니다.")

                    if volume < 40:
                        temp = nearest_orders.copy()

                        unassigned_points = unassigned_group_orders[['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, unassigned_points), axis=1)
                        
                        temp.sort_values('dist_to_A', inplace=True)
                        needed = 40 - volume
                        available = nearest_volume - 20  # 이동 가능한 최대 주문 수
                        move_count = min(needed, available)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                            break


                        # 이동하는 주문들의 driver 관련 컬럼을 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan   
                                                        
                        df_shipping_region.loc[orders_to_move.index, 'group'] = unassigned_max_group

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        if unassigned_max_group_volume >= 40:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']
                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                         
                                print(f"[4차 Y/R 배정] A 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                                assigned_non_orange += 1
                                leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break
                    
                    elif volume >= 56:
                        if nearest_group in group_centroids:
                            nearest_group_centroid = group_centroids[nearest_group]
                        else:
                            nearest_group_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
                            nearest_group_centroid = (nearest_group_orders['lat'].mean(), nearest_group_orders['lng'].mean())
                            group_centroids[nearest_group] = nearest_group_centroid

                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()

                        nearest_group_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]
                        temp['dist_to_A'] = temp.apply(lambda row: min_dist_to_group(row, nearest_group_points), axis=1)
                        
                        temp.sort_values('dist_to_A', inplace=True)
                        # 60이상은 위에서 이미 클러스터링 돼서 남아있는 그룹은 50~59
                        needed = volume - 55
                        available = 29 - nearest_volume  # 이동 가능한 최대 주문 수 => 배정이 안 될 수도 있으므로, 여유 둠.(최후)
                        move_count = min(needed, available)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [Y/R 4차 재배정]을 종료합니다.")
                            break


                        # (1) 이동하는 주문들의 driver 컬럼 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                        # (2) group을 nearest_group으로 변경
                        df_shipping_region.loc[orders_to_move.index, 'group'] = nearest_group

                         # (3) 만약 nearest_group에
                         # 버니(Y/R 등)가 이미 배정되어 있다면, 그 버니정보를 이동된 주문에도 적용
                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group)
                            & (df_shipping_region['driver_type'].notna())
                        ].head(1)

                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]
                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code
                        # 이후 물량 카운트, 버니 배정 처리
                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")


                        if unassigned_max_group_volume <= 55:
                            if assigned_non_orange < len(non_orange_drivers):
                                driver = non_orange_drivers.iloc[assigned_non_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                                new_name = f"{unassigned_max_group}_Y/R"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                print(f"[4차 Y/R 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")

                                assigned_non_orange += 1
                                leftover_non_orange = len(non_orange_drivers) - assigned_non_orange
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break
            
            else:
                print(f"근접 그룹 {nearest_group}의 driver_type({nearest_driver_type})에 대해 정의된 로직이 없습니다.")
        else:
            print("미배정 그룹 외에 다른 그룹이 없습니다.")

    # 이후 오렌지 버니 추가 배정 로직
    # 2차 ORANGE 재배정

    stop_all = False
    print("### [O 2차 재배정] ###")

    # 오렌지 드라이버 수만큼 반복 (안전장치 역할)
    for _ in range(len(orange_drivers)):
        if leftover_orange <= 0:
            stop_all = True
            break

        unassigned_df = df_shipping_region[df_shipping_region['driver_type'].isna()]
        
        # 현재 group별 주문 수 집계
        group_counts = unassigned_df.groupby('group')['shipping_uuid'].count()
        # 40건 이상인 그룹만 추출
        candidate_groups = group_counts[group_counts >= 40].index.tolist()

        print(f"[O 2차 재배정] 남은 오렌지 버니: {leftover_orange}명, 배정되지 않은 40건 이상 그룹: {candidate_groups}")

        # 40건 이상인 그룹이 없으면 종료
        if not candidate_groups:
            break

        # 40건 이상 그룹을 순회
        for grp in candidate_groups:
            if leftover_orange <= 0:
                stop_all = True
                break

            # 스택(LIFO)으로 관리
            group_stack = [grp]

            while group_stack and not stop_all:
                current_group = group_stack.pop()
                grp_orders = df_shipping_region[df_shipping_region['group'] == current_group].copy()
                current_count = grp_orders.shape[0]

                # 이미 40건 미만으로 줄었다면 스킵
                if current_count < 40:
                    continue

                print(f"[O 2차 재배정] 그룹[{current_group}] (주문수: {current_count}) → KMeans 2클러스터링 시도")

                # 좌표 변환
                grp_orders['lat_rad'] = grp_orders['lat'].apply(radians)
                grp_orders['lng_rad'] = grp_orders['lng'].apply(radians)
                coords = grp_orders[['lat_rad', 'lng_rad']].to_numpy()

                km = KMeans(n_clusters=2, init='k-means++', random_state=42)
                cluster_labels = km.fit_predict(coords)
                grp_orders['cluster'] = cluster_labels
                cluster_counts = grp_orders['cluster'].value_counts()

                if len(cluster_counts) < 2:
                    # 클러스터가 1개만 나오면(이상치) 다음으로
                    continue

                label_a = cluster_counts.index[0]
                count_a = cluster_counts.iloc[0]
                label_b = cluster_counts.index[1]
                count_b = cluster_counts.iloc[1]

                print(f"  → 클러스터 A={label_a}({count_a}건), 클러스터 B={label_b}({count_b}건)")

                # ---------------------
                # [조건1] 두 클러스터 모두 40건 이상
                # ---------------------
                if count_a >= 40 and count_b >= 40:
                    new_group_a = f"{current_group}_SPLIT_{label_a}"
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    idx_a = grp_orders[grp_orders['cluster'] == label_a].index
                    idx_b = grp_orders[grp_orders['cluster'] == label_b].index

                    df_shipping_region.loc[idx_a, 'group'] = new_group_a
                    df_shipping_region.loc[idx_b, 'group'] = new_group_b

                    print(f"  → 두 클러스터 모두 40건 이상 → [{new_group_a}], [{new_group_b}] 스택에 재추가")
                    group_stack.append(new_group_a)
                    group_stack.append(new_group_b)
                    continue

                # ---------------------
                # [조건2] 한쪽은 20~30, 다른 쪽은 40 이상
                # ---------------------
                if 20 <= count_a < 30 and count_b >= 40:
                    # A 즉시 배정, B는 스택 재추가
                    new_orange_group_a = f"{current_group}_O_{label_a}"
                    new_group_b = f"{current_group}_SPLIT_{label_b}"

                    idx_a = grp_orders[grp_orders['cluster'] == label_a].index
                    idx_b = grp_orders[grp_orders['cluster'] == label_b].index

                    df_shipping_region.loc[idx_a, 'group'] = new_orange_group_a
                    df_shipping_region.loc[idx_b, 'group'] = new_group_b

                    # 오렌지 배정
                    if assigned_orange < len(orange_drivers):
                        driver = orange_drivers.iloc[assigned_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_a, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_a, 'driver_code'] = driver['code']

                        assigned_orange += 1
                        leftover_orange = len(orange_drivers) - assigned_orange
                        print(f"  → [조건2] 그룹[{new_orange_group_a}]({count_a}건) → [{driver['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 배정할 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    # 남은 클러스터(B)는 스택에 추가
                    group_stack.append(new_group_b)
                    continue

                if 20 <= count_b < 30 and count_a >= 40:
                    # B 즉시 배정, A는 스택 재추가
                    new_orange_group_b = f"{current_group}_O_{label_b}"
                    new_group_a = f"{current_group}_SPLIT_{label_a}"

                    idx_b = grp_orders[grp_orders['cluster'] == label_b].index
                    idx_a = grp_orders[grp_orders['cluster'] == label_a].index

                    df_shipping_region.loc[idx_b, 'group'] = new_orange_group_b
                    df_shipping_region.loc[idx_a, 'group'] = new_group_a

                    # 오렌지 배정
                    if assigned_orange < len(orange_drivers):
                        driver = orange_drivers.iloc[assigned_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_b, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_b, 'driver_code'] = driver['code']
                        assigned_orange += 1
                        leftover_orange = len(orange_drivers) - assigned_orange
                        print(f"  → [조건2] 그룹[{new_orange_group_b}]({count_b}건) → 오렌지 버니[{driver['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 배정할 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    group_stack.append(new_group_a)
                    continue

                # ---------------------
                # [조건3] 두 클러스터 모두 20~30
                # ---------------------
                if (20 <= count_a < 30) and (20 <= count_b < 30):
                    new_orange_group_a = f"{current_group}_O_{label_a}"
                    new_orange_group_b = f"{current_group}_O_{label_b}"

                    idx_a = grp_orders[grp_orders['cluster'] == label_a].index
                    idx_b = grp_orders[grp_orders['cluster'] == label_b].index

                    df_shipping_region.loc[idx_a, 'group'] = new_orange_group_a
                    df_shipping_region.loc[idx_b, 'group'] = new_orange_group_b

                    # A 배정
                    if assigned_orange < len(orange_drivers):
                        driver_a = orange_drivers.iloc[assigned_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_a, 'driver_type'] = driver_a['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_a, 'driver_code'] = driver_a['code']

                        assigned_orange += 1
                        leftover_orange = len(orange_drivers) - assigned_orange
                        print(f"  → [조건3] 그룹[{new_orange_group_a}]({count_a}건) → 오렌지 버니[{driver_a['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    # B 배정
                    if assigned_orange < len(orange_drivers):
                        driver_b = orange_drivers.iloc[assigned_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_b, 'driver_type'] = driver_b['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_b, 'driver_code'] = driver_b['code']
                        assigned_orange += 1
                        leftover_orange = len(orange_drivers) - assigned_orange
                        print(f"  → [조건3] 그룹[{new_orange_group_b}]({count_b}건) → 오렌지 버니[{driver_b['Type']}] 배정")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    continue

                # ---------------------
                # [조건4] 한 클러스터가 20~30, 다른 클러스터가 30~40
                # ---------------------
                def in_range(x, low, high):
                    return (x >= low) and (x < high)

                # a=20~30, b=30~40
                if in_range(count_a, 20, 30) and in_range(count_b, 30, 40):
                    new_orange_group_a = f"{current_group}_O_{label_a}"
                    remain_group_b = f"{current_group}_remain_{label_b}"

                    idx_a = grp_orders[grp_orders['cluster'] == label_a].index
                    idx_b = grp_orders[grp_orders['cluster'] == label_b].index

                    df_shipping_region.loc[idx_a, 'group'] = new_orange_group_a
                    df_shipping_region.loc[idx_b, 'group'] = remain_group_b

                    # 20~30 클러스터 O 배정
                    if assigned_orange < len(orange_drivers):
                        driver = orange_drivers.iloc[assigned_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_a, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_a, 'driver_code'] = driver['code']
                        assigned_orange += 1
                        leftover_orange = len(orange_drivers) - assigned_orange
                        print(f"  → [조건4] 그룹[{new_orange_group_a}]({count_a}건) → [{driver['Type']}] 배정, 나머지[{remain_group_b}]는 유지({count_b}건)")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    continue

                # b=20~30, a=30~40
                if in_range(count_b, 20, 30) and in_range(count_a, 30, 40):
                    new_orange_group_b = f"{current_group}_O_{label_b}"
                    remain_group_a = f"{current_group}_remain_{label_a}"

                    idx_b = grp_orders[grp_orders['cluster'] == label_b].index
                    idx_a = grp_orders[grp_orders['cluster'] == label_a].index

                    df_shipping_region.loc[idx_b, 'group'] = new_orange_group_b
                    df_shipping_region.loc[idx_a, 'group'] = remain_group_a

                    # 20~30 클러스터 O 배정
                    if assigned_orange < len(orange_drivers):
                        driver = orange_drivers.iloc[assigned_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_b, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_orange_group_b, 'driver_code'] = driver['code']
                        assigned_orange += 1
                        leftover_orange = len(orange_drivers) - assigned_orange
                        print(f"  → [조건4] 그룹[{new_orange_group_b}]({count_b}건) → [{driver['Type']}] 배정, 나머지[{remain_group_a}]는 유지({count_a}건)")
                    else:
                        print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                        stop_all = True
                        break

                    continue

                # ---------------------
                # [조건5] 그 외 - 부족 / 초과 로직으로 20건 맞추기
                # ---------------------
                # 작은/큰 클러스터 식별
                if cluster_counts[label_a] <= cluster_counts[label_b]:
                    small_label, large_label = label_a, label_b
                else:
                    small_label, large_label = label_b, label_a

                small_count = cluster_counts[small_label]
                large_count = cluster_counts[large_label]

                print(f"  → [조건5] 작은 클러스터={small_label}({small_count}건), 큰 클러스터={large_label}({large_count}건)")

                # A) 작은 클러스터 < 20 → 큰 쪽에서 가져와 20 맞추기
                if small_count < 20:
                    deficit = 20 - small_count
                    print(f"    → 작은 클러스터 부족 {deficit}건. 큰 클러스터 -> 작은 클러스터 이동")
                    small_cluster_points = grp_orders[grp_orders['cluster'] == small_label][['lat','lng']]

                    large_orders = grp_orders[grp_orders['cluster'] == large_label].copy()
                    large_orders['dist_to_small'] = large_orders.apply(
                            lambda row: min_dist_to_group(row, small_cluster_points), axis=1
                    )

                    # 가장 작은 순(= 작은 클러스터에 가까운 순)으로 정렬
                    large_orders.sort_values('dist_to_small', inplace=True)

                    # deficit만큼 뽑아서 이동
                    indices_to_move = large_orders.index[:deficit]

                    # 이동
                    grp_orders.loc[indices_to_move, 'cluster'] = small_label

                    df_shipping_region.loc[indices_to_move, ['driver_type', 'driver_code']] = np.nan
                    print(f"  → {deficit}건을 대형 클러스터에서 작은 클러스터로 이동 (가장 가까운 순)")

                # B) 작은 클러스터 > 20 → 일부를 큰 쪽으로 이동
                elif small_count >= 30:
                    surplus = small_count - 20
                    print(f"    → 작은 클러스터 초과 {surplus}건. 작은 클러스터 -> 큰 클러스터 이동")

                    # 큰 클러스터의 모든 점
                    large_cluster_points = grp_orders[grp_orders['cluster'] == large_label][['lat','lng']]

                    # 이동 후보 = 작은 클러스터의 주문들
                    small_orders = grp_orders[grp_orders['cluster'] == small_label].copy()

                    small_orders['dist_to_large'] = small_orders.apply(
                        lambda row: min_dist_to_group(row, large_cluster_points), axis=1
                    )
                    small_orders.sort_values('dist_to_large', inplace=True)

                    # surplus만큼을 큰 클러스터로 이동
                    indices_to_move = small_orders.index[:surplus]

                    grp_orders.loc[indices_to_move, 'cluster'] = large_label
                    
                    df_shipping_region.loc[indices_to_move, ['driver_type', 'driver_code']] = np.nan
                    
                    print(f"  → {surplus}건을 작은 클러스터에서 대형 클러스터로 이동")
                    
                new_counts = grp_orders['cluster'].value_counts()
                new_small_count = new_counts.get(small_label, 0)
                print(f"  → 조정 후 작은 클러스터 {small_label} 주문 수: {new_small_count} (목표:20)")

                if new_small_count == 20:
                    # base_name은 원래 그룹명에서 클러스터 관련 문자열 제거 (있다면)
                    base_name = grp.split('_cluster')[0]
                    yr_group_counter.setdefault(base_name, 0)
                    yr_group_counter[base_name] += 1
                    new_grp_label_target = f"{base_name}_cluster_O_{yr_group_counter[base_name]}"
                    new_grp_label_remaining = f"{grp}_remaining"
                    indices_target = grp_orders[grp_orders['cluster'] == small_label].index
                    indices_remaining = grp_orders[grp_orders['cluster'] == large_label].index

                    df_shipping_region.loc[indices_target, 'group'] = new_grp_label_target
                    df_shipping_region.loc[indices_remaining, 'group'] = new_grp_label_remaining

                    if assigned_orange < len(orange_drivers):
                        driver = orange_drivers.iloc[assigned_orange]
                        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label_target, 'driver_type'] = driver['Type']
                        df_shipping_region.loc[df_shipping_region['group'] == new_grp_label_target, 'driver_code'] = driver['code']

                        print(f"[2차 O 재배정] 그룹[{new_grp_label_target}] (주문수: {new_small_count}) → [{driver['Type']}] 배정")
                        assigned_orange += 1
                        leftover_orange = len(orange_drivers) - assigned_orange
                        group_centroids = recalc_group_centroids(df_shipping_region)

                        print("[2차 O 재배정] 후 그룹별 물량:\n", df_shipping_region.groupby('group')['shipping_uuid'].count())

                    else:
                        print("[2차 O 재배정] 배정 가능한 오렌지 버니가 더 이상 없습니다.")
                        break

                if leftover_orange <= 0:
                    print("[2차 O 재배정] 더 이상 오렌지 버니가 없습니다.")
                    stop_all = True
                    break

        leftover_orange = len(orange_drivers) - assigned_orange
        print(f"[O 2차 재배정] 루프 종료, 남은 오렌지 버니: {leftover_orange}")

        if stop_all:
            break

    print("[O 2차 재배정] 로직 종료 또는 버니 부족")

            
    # [3차 O 재배정]
    while leftover_orange > 0:    
        # 남은 20-29 지역 남은 오렌지 버니에게 배당
        plus_unassigned_groups = df_shipping_region[df_shipping_region['driver_type'].isna()]['group'].unique()
        plus_candidate_groups = []
        for grp in plus_unassigned_groups:
            cnt = df_shipping_region[df_shipping_region['group'] == grp]['shipping_uuid'].count()
            if 20 <= cnt <= 29 :
                plus_candidate_groups.append(grp)

        if not plus_candidate_groups:
            print("[3차 O 재배정] 더 이상 주문 수 20이상 29 이하의 미배정 그룹이 없습니다.")
            break

        print(f"[3차 O 재배정]대상 그룹: {plus_candidate_groups}")

        for grp in plus_candidate_groups:

            if leftover_orange <= 0:
                break
            
            grp_orders = df_shipping_region[df_shipping_region['group'] == grp]
            print("[3차 O 재배정] 20~29 그룹 Orange 에게 배정")
            new_grp_label = f"{grp}_O"
            selected_idx = grp_orders.index
            df_shipping_region.loc[selected_idx, 'group'] = new_grp_label

            if assigned_orange < len(orange_drivers):
                driver = orange_drivers.iloc[assigned_orange]
                df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_type'] = driver['Type']
                df_shipping_region.loc[df_shipping_region['group'] == new_grp_label, 'driver_code'] = driver['code']

                print(f"[3차 O 재배정] 그룹[{new_grp_label}] (물량={len(grp_orders)}) → [{driver['Type']}] 배정")
                assigned_orange += 1
                leftover_orange = len(orange_drivers) - assigned_orange
                group_centroids = recalc_group_centroids(df_shipping_region)
                if grp in group_centroids:
                    del group_centroids[grp]
        

    # [4차 O 재배정]
    print("[ORANGE 4차 재배정]")
    while leftover_orange > 0:

        unassigned_groups = df_shipping_region[df_shipping_region['driver_type'].isna()]['group'].unique()

        if len(unassigned_groups) == 0:
            print("[ORANGE 4차 재배정] 배정되지 않은 그룹이 없습니다.")
            break
        unassigned_group_counts = df_shipping_region[df_shipping_region['group'].isin(unassigned_groups)] \
                                    .groupby('group')['shipping_uuid'].count()
        
        # 버니가 배정되지 않은 그룹 중 물량이 가장 많은 그룹
        unassigned_max_group = unassigned_group_counts.idxmax()
        print(f"선택된 unassigned_max_group (물량 많은 그룹): {unassigned_max_group} (물량: {unassigned_group_counts[unassigned_max_group]}건)")

        unassigned_group_orders = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]
         # - 목적지/출발지에 쓰일 points
        unassigned_points = unassigned_group_orders[['lat','lng']]

        plus_candidate_groups = [g for g in group_centroids.keys() if g != unassigned_max_group]
        if plus_candidate_groups:
            distances = {}
            for g in plus_candidate_groups:
                centroid = group_centroids[g]

                dist = np.sqrt((unassigned_points['lat'].mean() - centroid[0])**2 +
                       (unassigned_points['lng'].mean() - centroid[1])**2)
                distances[g] = dist

            nearest_group = min(distances, key=distances.get)
            print(f"unassigned_max_group 과 가장 가까운 그룹: {nearest_group}")

            nearest_orders = df_shipping_region[df_shipping_region['group'] == nearest_group]
            
            if not nearest_orders.empty:
                nearest_driver_type = nearest_orders['driver_type'].iloc[0]
            else:
                nearest_driver_type = None

            print(f"근접 그룹 {nearest_group}의 driver_type: {nearest_driver_type}")
            
            nearest_points = nearest_orders[['lat','lng']]

            if nearest_group in group_centroids:
                nearest_group_centroid = group_centroids[nearest_group]
            else:
                nearest_group_centroid = (nearest_orders['lat'].mean(), nearest_orders['lng'].mean())
                group_centroids[nearest_group] = nearest_group_centroid

            # 배정되지 않은 그룹 수량
            volume = unassigned_group_counts[unassigned_max_group]

            # 근접 그룹 수량
            nearest_volume = len(nearest_orders)

            # 근접 그룹의 드라이버 타입이 NULL일 때
            if pd.isna(nearest_driver_type):
                print(f"근접 그룹 {nearest_group}의 driver_type이 null이므로, unassigned_max_group 의 물량이 20이 될 때까지 데이터를 이동합니다.")

                # 배정되지 않은 그룹의 수량이 20 미만일때(배정되지 않은 그룹의 중심점에서 가까운 순서대로 needed 만큼 nearest_group에서 가져오기)
                if volume < 20:
                    needed = 20 - volume
                    available = nearest_volume - 20  # 이동 가능한 최대 주문 수
                    move_count = min(needed, available)
                    print(f"unassigned_max_group={unassigned_max_group} 부족분={needed}, 근접그룹={nearest_group} (volume={nearest_volume})에서 가져올 수={move_count}")

                    if  needed > available:
                        print("이동할 데이터가 없으므로, [ORANGE 4차 재배정]을 종료합니다.")
                        break

                    temp = nearest_orders.copy()
                    temp['dist_to_A'] = temp.apply(
                        lambda r: min_dist_to_group(r, unassigned_points),
                        axis=1
                    )
                    temp.sort_values('dist_to_A', inplace=True)

                    orders_to_move = temp.head(move_count)

                    # 이미 배정된 버니정보가 있을 수도 있으므로 초기화
                    df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                    # 데이터 이동
                    df_shipping_region.loc[orders_to_move.index, 'group'] = unassigned_max_group
                    
                    # 베정되지 않은 그룹의 수량이 20되면 버니 배정
                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()

                    print(f"→ 이동 후 unassigned_max_group={unassigned_max_group}의 물량={unassigned_max_group_volume}건")

                    if unassigned_max_group_volume >= 20:
                        if assigned_orange < len(orange_drivers):
                            driver = orange_drivers.iloc[assigned_orange]
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                            new_name = f"{unassigned_max_group}_O"
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                            print(f"[ORANGE 4차 재배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, 남은 O 버니[{driver['uuid']}]에 배정합니다.")
                            assigned_orange += 1
                            leftover_orange = len(orange_drivers) - assigned_orange
                            group_centroids = recalc_group_centroids(df_shipping_region)
                    else:
                        print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                        break


                # 배정되지 않은 그룹의 물량이 30이상이라 ORANGE버니가 배정이 안될 때
                elif volume >= 30:
                    needed = volume - 29
                    temp = unassigned_group_orders.copy()
                    temp['dist_to_A'] = temp.apply(
                        lambda r: min_dist_to_group(r, nearest_points),
                        axis=1
                    )

                    temp.sort_values('dist_to_A', inplace=True)

                    orders_to_move = temp.head(needed)

                    print(f"nearest_group 에 이동할 주문 수: {len(orders_to_move)}")

                    if len(orders_to_move) == 0:
                        print("이동할 데이터가 없으므로 종료")
                        break

                    # 이미 배정된 버니 정보가 있을 수도 있으므로 초기화
                    df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan  

                    # nearest_group으로 데이터 이동
                    df_shipping_region.loc[orders_to_move.index, 'group'] = nearest_group
                    
                    
                    unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                    print(f"→ 이동 후 unassigned_max_group={unassigned_max_group}={unassigned_max_group_volume}건, nearest_group={nearest_group}={df_shipping_region[df_shipping_region['group'] == nearest_group].shape[0]}건")

                    # unassigned_max_group_volume이 29가 되면 버니 배정
                    if unassigned_max_group_volume <= 29:
                        if assigned_orange < len(orange_drivers):
                            driver = orange_drivers.iloc[assigned_orange]
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                            new_name = f"{unassigned_max_group}_O"
                            df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name
                            print(f"[ORANGE 4차 재배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                            assigned_orange += 1
                            leftover_orange = len(orange_drivers) - assigned_orange
                            group_centroids = recalc_group_centroids(df_shipping_region)
                 
               
            
            elif nearest_driver_type in ['YELLOW', 'RAINBOW']:
                # 근접 그룹(Y/R 그룹)의 물량이 40 미만이면 안된다.
                if nearest_volume < 40:
                    print(f"근접 그룹 {nearest_group}의 물량이 40 미만({nearest_volume}건)이라 unassigned_max_group 에 데이터를 가져올 수 없습니다. 종료합니다.")

                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 Y/R입니다. unassigned_max_group 목표 물량: 20~29, 데이터 이동 시도합니다.")

                    if volume < 20:
                        needed = 20 - volume
                        available = nearest_volume - 40  # 이동 가능한 최대 주문 수
                        move_count = min(needed, available)
                        print(f"unassigned_max_group={unassigned_max_group} 부족분={needed}, 근접그룹={nearest_group} (volume={nearest_volume})에서 가져올 수={move_count}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [ORANGE 4차 재배정]을 종료합니다.")
                            break

                        temp = nearest_orders.copy()
                        temp['dist_to_A'] = temp.apply(
                            lambda r: min_dist_to_group(r, unassigned_points),
                            axis=1
                        )
                        temp.sort_values('dist_to_A', inplace=True)

                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        if  needed > available:
                            print("이동할 데이터가 없으므로, [ORANGE 4차 재배정]을 종료합니다.")

                            break

                        # 이동하는 주문들의 driver 관련 컬럼을 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                        df_shipping_region.loc[orders_to_move.index, 'group'] = unassigned_max_group

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        if unassigned_max_group_volume >= 20:
                            if assigned_orange < len(orange_drivers):
                                driver = orange_drivers.iloc[assigned_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                                new_name = f"{unassigned_max_group}_O"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                print(f"[ORANGE 4차 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                                
                                assigned_orange += 1
                                leftover_orange = len(orange_drivers) - assigned_orange
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")
                            break

                    # nearest 그룹(Y/R)에 넘겨주는 케이스
                    elif volume >= 30:
                        needed = volume - 29
                        available = 55 - nearest_volume 
                        move_count = min(needed, available)
                        
                        if needed > available:
                            print("이동할 데이터가 없으므로, [O 4차 재배정]을 종료합니다.")
                            break

                        # 이동 후보 = unassigned_max_group의 주문들
                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()

                        # 'nearest_group' 내 모든 점
                        nearest_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]

                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, nearest_points), axis=1)
                        temp.sort_values('dist_to_A', inplace=True)

                        orders_to_move = temp.head(move_count)
                        print(f"nearest_group 에 이동할 주문 수: {len(orders_to_move)}")


                        # 이동하는 주문들의 driver 컬럼 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                        # group을 nearest_group으로 변경
                        df_shipping_region.loc[orders_to_move.index, 'group'] = nearest_group

                         # (3) 만약 nearest_group에 버니(Y/R 등)가 이미 배정되어 있다면, 그 버니 정보를 이동된 주문에도 적용
                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group)
                            & (df_shipping_region['driver_type'].notna())
                        ].head(1)

                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]
                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code
                        # 이후 물량 카운트, 버니 배정 처리
                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")


                        if unassigned_max_group_volume <= 29:
                            if assigned_orange < len(orange_drivers):
                                driver = orange_drivers.iloc[assigned_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                                new_name = f"{unassigned_max_group}_O"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                print(f"[4차 ORANGE 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")

                                assigned_orange += 1
                                leftover_orange = len(orange_drivers) - assigned_orange
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")

            elif nearest_driver_type == 'ORANGE':
                # 근접 그룹(Orange 그룹)의 물량이 20 미만이면 안된다.
                if nearest_volume < 20:
                    
                    print(f"근접 그룹 {nearest_group}의 물량이 20 미만({nearest_volume}건)이라 unassigned_max_group 에 데이터를 가져올 수 없습니다. 종료합니다.")
                    break
                else:
                    print(f"근접 그룹 {nearest_group}의 driver_type이 Orange입니다. unassigned_max_group 의 물량이 20이 될 때까지 데이터 이동 시도합니다.")

                    if volume < 20:
                        needed = 20 - volume
                        available = nearest_volume - 20
                        move_count = min(needed, available)
                        
                        if needed > available:
                            print("이동할 데이터가 없으므로, [O 4차 재배정] 종료.")
                            break

                        temp = nearest_orders.copy()
                        unassigned_points = df_shipping_region[df_shipping_region['group'] == unassigned_max_group][['lat','lng']]

                        temp['dist_to_A'] = temp.apply(
                            lambda row: min_dist_to_group(row, unassigned_points),
                            axis=1
                        )
                        temp.sort_values('dist_to_A', inplace=True)
                        orders_to_move = temp.head(move_count)
                        print(f"unassigned_max_group 에 이동할 주문 수: {len(orders_to_move)}")

                        # 이동하는 주문들의 driver 관련 컬럼을 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan   
                                                        
                        df_shipping_region.loc[orders_to_move.index, 'group'] = unassigned_max_group

                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()

                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")


                        if unassigned_max_group_volume >= 20:
                            if assigned_orange < len(orange_drivers):
                                driver = orange_drivers.iloc[assigned_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                                new_name = f"{unassigned_max_group}_O"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                         
                                print(f"[4차 ORANGE 배정] A 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")
                                assigned_orange += 1
                                leftover_orange = len(orange_drivers) - assigned_orange    
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")

                    # nearest 그룹(O)에 넘겨주는 케이스
                    elif volume >= 30:
                        needed = volume - 29
                        available = 55 - nearest_volume 
                        move_count = min(needed, available)
                        
                        if needed > available:
                            print("이동할 데이터가 없으므로, [O 4차 재배정]을 종료합니다.")
                            break

                        # 이동 후보 = unassigned_max_group의 주문들
                        temp = df_shipping_region[df_shipping_region['group'] == unassigned_max_group].copy()

                        # 'nearest_group' 내 모든 점
                        nearest_points = df_shipping_region[df_shipping_region['group'] == nearest_group][['lat','lng']]

                        temp['dist_to_A'] = temp.apply(lambda r: min_dist_to_group(r, nearest_points), axis=1)
                        temp.sort_values('dist_to_A', inplace=True)

                        orders_to_move = temp.head(move_count)
                        print(f"nearest_group 에 이동할 주문 수: {len(orders_to_move)}")

                        # 이동하는 주문들의 driver 컬럼 초기화
                        df_shipping_region.loc[orders_to_move.index, ['driver_type', 'driver_code']] = np.nan

                        # group을 nearest_group으로 변경
                        df_shipping_region.loc[orders_to_move.index, 'group'] = nearest_group

                         # (3) 만약 nearest_group에 버니(O)가 이미 배정되어 있다면, 버니 정보를 이동된 주문에도 적용
                        driver_data = df_shipping_region.loc[
                            (df_shipping_region['group'] == nearest_group)
                            & (df_shipping_region['driver_type'].notna())
                        ].head(1)

                        if not driver_data.empty:
                            assigned_type = driver_data['driver_type'].iloc[0]
                            assigned_code = driver_data['driver_code'].iloc[0]

                            df_shipping_region.loc[orders_to_move.index, 'driver_type'] = assigned_type
                            df_shipping_region.loc[orders_to_move.index, 'driver_code'] = assigned_code

                        # 이후 물량 카운트, 버니 배정 처리
                        unassigned_max_group_volume = df_shipping_region[df_shipping_region['group'] == unassigned_max_group]['shipping_uuid'].count()
                        nearest_volume = df_shipping_region[df_shipping_region['group'] == nearest_group]['shipping_uuid'].count()
                        print(f"이동 후, unassigned_group({unassigned_max_group})={unassigned_max_group_volume}건, nearest_group({nearest_group})={nearest_volume}건")

                        if unassigned_max_group_volume <= 29:
                            if assigned_orange < len(orange_drivers):
                                driver = orange_drivers.iloc[assigned_orange]
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_type'] = driver['Type']
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'driver_code'] = driver['code']

                                new_name = f"{unassigned_max_group}_O"
                                df_shipping_region.loc[df_shipping_region['group'] == unassigned_max_group, 'group'] = new_name                                                      
                                print(f"[4차 ORANGE 배정] 그룹({new_name})의 주문 수가 {unassigned_max_group_volume}건이 되어, [{driver['Type']}]에 배정합니다.")

                                assigned_orange += 1
                                leftover_orange = len(orange_drivers) - assigned_orange
                                group_centroids = recalc_group_centroids(df_shipping_region)
                        else:
                            print(f"이동할 수 있는 물량이 부족해 종료, 필요물량: {needed}, 가져올 수 있는 물량:{move_count}")

            else:
                print(f"근접 그룹 {nearest_group}의 driver_type({nearest_driver_type})에 대해 정의된 로직이 없습니다.")
        else:
            print("미배정 그룹 외에 다른 그룹이 없습니다.")
        



    used_drivers_idx = list(non_orange_drivers.index[:assigned_non_orange]) + list(orange_drivers.index[:assigned_orange])
    leftover_driver_df = fix_region_workflow_day_bunny_df.drop(used_drivers_idx)
    return df_shipping_region, leftover_driver_df


In [169]:

# ---------------------------------------------------------------------------
# 4) 지역별 처리
# ---------------------------------------------------------------------------
def process_region(
    region_name,
    zipcode_groups,
    df_shipping,
    workflow_day_bunny_df,
    leftover_all_fix_drivers,
):

    print(f"### {region_name} 처리 시작 ###")

    zipcode_groups = zipcode_groups[zipcode_groups['region']==region_name]

    # (2) 그룹화 및 데이터 저장 
    result_gdf, df_shipping_region = group_and_map_zipcodes(zipcode_groups, df_shipping)

    # 지역별 bunny 기사 필터링
    region_workflow_day_bunny_df = workflow_day_bunny_df[workflow_day_bunny_df['Area'] == df_shipping_region.iloc[0, 3]].copy()

    # (3) 고정 기사(YELLOW/RAINBOW/ORANGE) & 화이트(WHITE) 기사 분리
    fix_region_workflow_day_bunny_df = region_workflow_day_bunny_df[
        region_workflow_day_bunny_df['Type'].isin(['YELLOW','RAINBOW','ORANGE'])
    ].copy()

    # 버니에게 배정 전 그룹별 물량
    df_shipping_region_group_count=df_shipping_region.groupby('group')['shipping_uuid'].agg('count').reset_index()
    print(f"{region_name} 초기 그룹별 물량\n{df_shipping_region_group_count}")

    fix_drivers = len(fix_region_workflow_day_bunny_df)

    if fix_drivers > 0:
        # 고정기사 배정
        df_shipping_region, leftover_driver_df = assign_fixed_drivers(df_shipping_region, fix_region_workflow_day_bunny_df)

        # leftover_all_fix_drivers 누적
        if not leftover_driver_df.empty:
            leftover_driver_df = leftover_driver_df.copy()
            leftover_driver_df['Region'] = region_name
            leftover_all_fix_drivers = pd.concat([leftover_all_fix_drivers, leftover_driver_df], ignore_index=True)

    # 미리 driver 컬럼이 없으면 생성
    for col in ['driver_type', 'driver_code']:
        if col not in df_shipping_region.columns:
            df_shipping_region[col] = np.nan

    unassigned_white_groups = df_shipping_region.groupby('group') \
        .filter(lambda x: x['driver_type'].isna().all()) \
        ['group'].unique()

    print("화이트 처리 대상 그룹:", unassigned_white_groups)
    for grp in unassigned_white_groups:
        df_shipping_region = isolate_white_clusters(df_shipping_region, grp)
        # 최종 결과 출력: 화이트 처리 후 그룹별 물량
        print("\n[화이트 처리 후 그룹별 물량]\n", df_shipping_region.groupby('group')['shipping_uuid'].count())
        
    print(f"{region_name} 처리 완료\n")
    return df_shipping_region, result_gdf, leftover_all_fix_drivers


In [170]:

# ---------------------------------------------------------------------------
# 5) 최종 실행 함수
# ---------------------------------------------------------------------------
def run_clustering(
    zipcode_groups,
    workflow_day_shipping_items_df,
    workflow_day_bunny_df,
):
    """
    전체 지역(zipcode_groups) 순회하며
    1) process_region() 호출 -> 기사 배정
    2) 최종적으로 지역별 DataFrame 통합
    3) leftover 기사를 별도 DataFrame에 저장
    4) 시각화 결과 HTML 생성 (visualize_clusters)
    """

    # 통합할 DF
    all_filtered_geo_dfs = pd.DataFrame()
    all_KMeans_dfs = pd.DataFrame()

    # 고정기사 DF
    leftover_all_fix_drivers = pd.DataFrame()


    # 원본 df_shipping 복사
    df_shipping = workflow_day_shipping_items_df.copy()

    # 지역별 처리
    for region_name in zipcode_groups['region'].unique():
        (
            df_shipping_region, 
            filtered_geo_df, 
            leftover_all_fix_drivers,
        ) = process_region(
            region_name=region_name,
            zipcode_groups=zipcode_groups,
            df_shipping=df_shipping,
            workflow_day_bunny_df=workflow_day_bunny_df,
            leftover_all_fix_drivers=leftover_all_fix_drivers,
        )

        # 처리된 주문은 원본에서 제거 (이미 배정된 주문)
        df_shipping = df_shipping[~df_shipping['shipping_uuid'].isin(df_shipping_region['shipping_uuid'])]

        # 지역별 결과 통합
        all_filtered_geo_dfs = pd.concat([all_filtered_geo_dfs, filtered_geo_df], ignore_index=True)
        all_KMeans_dfs = pd.concat([all_KMeans_dfs, df_shipping_region], ignore_index=True)

    # 남은 미배정 주문(df_shipping)도 합침
    all_KMeans_dfs = pd.concat([all_KMeans_dfs, df_shipping], ignore_index=True)


    # 그룹이 있는데 기사정보가 없는 행은 driver_type='WHITE'로 지정
    mask = (~all_KMeans_dfs['group'].isna()) & (all_KMeans_dfs['driver_type'].isna())
    all_KMeans_dfs.loc[mask, 'driver_type'] = 'WHITE'

    # 그룹이 없고, 기사정보가 없는 행은 driver_type = 'BLUE'로 지정 => 벌크
    mask1 = (all_KMeans_dfs['group'].isna()) & (all_KMeans_dfs['driver_type'].isna())
    all_KMeans_dfs.loc[mask1, 'driver_type'] = 'BLUE'

    # 1. driver_type 매핑 문자 정의
    type_mapping = {
        'WHITE': 'W',
        'RAINBOW': 'R',
        'YELLOW': 'Y',
        'ORANGE': 'O'
        # 나머지는 'B'로 처리
    }

    # 2. (Area, driver_type, group)가 모두 있는 행만 추출 (NaN은 제외)
    unique_rows = (
        all_KMeans_dfs[['Area', 'driver_type', 'group']]
        .dropna(subset=['Area','driver_type','group'])
        .drop_duplicates()
    )

    # 3. Area별 + driver_type별 순번을 관리할 dict
    dict_area_type_seq = {}  

    # 4. 최종 라벨 매핑 딕셔너리: dict_of_labels[(Area, driver_type, group)] = "강남W1" 등
    dict_of_labels = {}

    # unique_rows 순회하면서 순번 부여
    for _, row in unique_rows.iterrows():
        area_val = row['Area']
        d_type = row['driver_type']
        grp = row['group']

        # (Area, driver_type) 별로 seq 1부터 시작
        key_area_type = (area_val, d_type)
        if key_area_type not in dict_area_type_seq:
            dict_area_type_seq[key_area_type] = 1
        else:
            dict_area_type_seq[key_area_type] += 1

        seq = dict_area_type_seq[key_area_type]

        # driver_type이 매핑 사전에 없으면 'B'
        letter = type_mapping.get(d_type, 'B1')
        # 실제 라벨 형식: "강남-W1"
        label = f"{area_val}{letter}{seq}"

        # 해당 그룹(Area, driver_type, group)에 라벨 저장
        dict_of_labels[(area_val, d_type, grp)] = label

    # 각 행에서 (Area, driver_type, group)으로 dict_of_labels 조회
    for i in all_KMeans_dfs.index:
        area_val = all_KMeans_dfs.at[i, 'Area']
        d_type   = all_KMeans_dfs.at[i, 'driver_type']
        grp      = all_KMeans_dfs.at[i, 'group']

        key = (area_val, d_type, grp)
        # 매핑 딕셔너리에 있으면 그 라벨을, 없으면 기존 값을 유지
        if key in dict_of_labels:
            all_KMeans_dfs.at[i, 'cluster_label'] = dict_of_labels[key]
        else:
            if pd.notna(area_val) and pd.notna(d_type):
                letter = type_mapping.get(d_type, 'B1')
                all_KMeans_dfs.at[i, 'cluster_label'] = f"{area_val}{letter}"

    # leftover 기사 출력
    print("\n### 전체 지역에서 '배정받지 못한 고정 기사' 모음 ###")
    if leftover_all_fix_drivers.empty:
        print("모든 고정 기사 배정 완료(남은 기사 없음)")
    else:
        print(leftover_all_fix_drivers)


    return all_KMeans_dfs, leftover_all_fix_drivers


In [171]:
def is_weekend(year, month, day):
    return calendar.weekday(year, month, day) >= 5

def normalize_region(region):
    """
    workflow_day_bunny_df의 Area는 대부분 뒤에 '구'나 '시'가 없는 형식임.
    단, '일산서구', '일산동구'는 예외로 그대로 사용.
    """
    if isinstance(region, list):
        return [normalize_region(r) for r in region]
    
    if region in ["일산서구", "일산동구", "중구"]:
        return region
    if region.endswith("구") or region.endswith("시"):
        return region[:-1]
    return region
# ---------------------------------------------------------------------------
# 6) 실제 실행 
# ---------------------------------------------------------------------------
if __name__ == "__main__":


    workflow_day_shipping_items_df = workflow_day_shipping_items_df.copy()
    workflow_day_bunny_df = workflow_day_bunny_df.copy()
    
    df_shworkflow_day_shipping_items_dfipping = workflow_day_shipping_items_df.astype({'zipcode': str})
    workflow_day_shipping_items_df['zipcode'] = workflow_day_shipping_items_df['zipcode'].astype(str).str.zfill(5) 
    

    date = workflow_day_bunny_df['Date'].drop_duplicates()
    year, month, day = map(int, date[0].split('-'))

    # BLUE 기사들의 Area 목록 추출
    blue_areas = list(workflow_day_bunny_df.loc[workflow_day_bunny_df['Type'] == 'BLUE', 'Area'])
   

    if is_weekend(year, month, day):
        df_regular['region'] = df_weekend['region'].apply(normalize_region)
        chosen_df_weekend = df_weekend[~df_weekend['region'].isin(blue_areas)]

        chosen_zipcode_groups = chosen_df_weekend
        chosen_zipcode_groups.reset_index(drop=True, inplace=True)
        
    else:
        df_regular['region'] = df_regular['region'].apply(normalize_region)
        chosen_df_regular = df_regular[~df_regular['region'].isin(blue_areas)]

        chosen_zipcode_groups = chosen_df_regular
        chosen_zipcode_groups.reset_index(drop=True, inplace=True)
        
    
    all_KMeans_dfs, leftover_drivers = run_clustering(
        zipcode_groups=chosen_zipcode_groups,
        workflow_day_shipping_items_df=workflow_day_shipping_items_df,
        workflow_day_bunny_df=workflow_day_bunny_df,
    )

    # 기사가 부족해 배정되지 않은 지역
    all_KMeans_dfs_left = all_KMeans_dfs[(~all_KMeans_dfs['group'].isna()) & (all_KMeans_dfs['driver_type'].isna())].reset_index(drop=True)

    # 드라이버 할당 되지 않은 그룹들중 20개 미만인 지역
    all_KMeans_dfs_sam = all_KMeans_dfs.groupby('group').agg({'shipping_uuid':'count', 'driver_type' : 'count'}).reset_index()
    all_KMeans_dfs_sam_20less=all_KMeans_dfs_sam[all_KMeans_dfs_sam['driver_type']==0]
    KMeans_dfs_none_driver_20less_group = all_KMeans_dfs_sam_20less[all_KMeans_dfs_sam_20less['shipping_uuid']<20]
    
    # 모든 그룹들 확인
    all_KMeans_dfs_20less_group = all_KMeans_dfs_sam[all_KMeans_dfs_sam['shipping_uuid']<20]

    # 결과 확인
    print("\n=== 벌크 지역 ===")
    print(blue_areas)
    print("\n=== 배정받지 못한 고정 기사 ===")
    print(leftover_drivers)
    print("\n=== 배정되지 않은 그룹들 중 물량 20개 미만인 지역 ===")
    print(KMeans_dfs_none_driver_20less_group)
    print("\n=== 전체 그룹들 중 물량 20개 미만인 지역 ===")
    print(all_KMeans_dfs_20less_group)

### 용산 처리 시작 ###
용산 초기 그룹별 물량
  group  shipping_uuid
0     A             84
1     B             54
2     C             33
[Y/R 버니 수] = 1
[ORANGE 버니 수] = 0
[Y/R 1차 배정] 40~50건 그룹 배정 후 남은 Yellow/Rainbow 버니 = 1
[O 1차 배정] 오렌지 버니 배정 후 남은 버니 = 0
### [Y/R 2차 재배정] ###
배정되지 않은 그룹 ['A']
[Y/R 2차 재배정] 남은 Y/R 버니: 1명
[Y/R 2차 재배정] 그룹 [A] (주문수: 84)에서 클러스터 추출 시도
  → 그룹 [A] 클러스터 결과: 클러스터1 0 (59건), larger 클러스터 1 (25건)
  → 초과: 19건 제거 필요
  → 19건을 큰 클러스터에서 작은 클러스터로 이동
  → 조정 후: 큰 클러스터 0 주문수: 40 (목표:40)
[Y/R 2차 재배정] 그룹[A_cluster_Y/R_1] (약 40건) → [YELLOW] 배정
[Y/R 2차 재배정] 후 그룹별 물량:
 group
A_cluster_Y/R_1    40
A_remaining        44
B                  54
C                  33
Name: shipping_uuid, dtype: int64
[Y/R 2차 재배정] 남은 Y/R 버니: 0명
[Y/R 2차 재배정] 로직 완료 or 버니 부족
### [Y/R 3차 재배정] ###
남은 버니 부족으로 종료
### [Y/R 4차 재배정] ###
남은 버니 부족으로 종료
### [O 2차 재배정] ###
[O 2차 재배정] 로직 종료 또는 버니 부족
[ORANGE 4차 재배정]
화이트 처리 대상 그룹: ['B' 'C' 'A_remaining']

[화이트 처리] 그룹 B 총 주문 수: 54건. 클러스터링 시도...
클러스터링 결과: {0: 29, 1: 25}
클러스터 초기 그룹 물량 20건이상

In [172]:
all_KMeans_dfs=all_KMeans_dfs.drop(columns='group')

In [173]:
# 각 그룹에 물량 집어넣기
# 각 실제 데이터(point)에 대해서 (lat, lng) 거리를 구하고, 가장 가까운 한 개 row의 group을 할당(가장 가까운 데이터의 group에 포함)

def overlap(overlap_df, all_KMeans_dfs):
    
    overlap_df = overlap_df.astype({'zipcode': str})
    overlap_df['zipcode'] = overlap_df['zipcode'].astype(str).str.zfill(5) 

    for col in ['driver_type', 'driver_code', 'cluster_label']:
        if col not in overlap_df.columns:
            overlap_df[col] = np.nan
            
    # 오버랩 데이터를 가장 가까이 있는 데이터의 그룹에 포함시키기
    for i, order_row in overlap_df.iterrows():
        area = order_row["Area"]
        lat_val = order_row["lat"]
        lng_val = order_row["lng"]
        
        
        # 1) Area가 같은 행만 추출
        area_slice = all_KMeans_dfs[all_KMeans_dfs["Area"] == area]
        
        # 2) 해당 지역(Area)에 데이터 자체가 전혀 없는 경우: 미배정 처리
        if area_slice.empty:
            overlap_df.at[i, "driver_type"]  = np.nan
            overlap_df.at[i, "driver_code"]  = np.nan
            overlap_df.at[i, "cluster_label"]  = np.nan
            
            print(f"[INFO] {area} 지역의 주문({order_row['shipping_uuid']})은 같은 지역 데이터가 없어 미배정 처리.")
            continue
        
        # 3) (lat, lng) 거리계산 -> 가장 가까운 group 찾기
        #    여기서는 유클리드로 비교
        area_slice["dist"] = np.sqrt((area_slice["lat"] - lat_val)**2 + 
                                     (area_slice["lng"] - lng_val)**2)
        
        # 가장 작은 dist를 갖는 row의 인덱스
        min_idx = area_slice["dist"].idxmin()
        nearest_row = area_slice.loc[min_idx]
        
        best_group_driver_type = nearest_row["driver_type"]
        
        # 그룹이 없을 경우 클러스터라벨 및 driver_type 추가하기
        cluster_label = all_KMeans_dfs[all_KMeans_dfs['Area'] == area][['cluster_label']].drop_duplicates()
        cluster_label = cluster_label.iloc[0]

        driver_type = all_KMeans_dfs[all_KMeans_dfs['Area'] == area][['driver_type']].drop_duplicates()
        driver_type = driver_type.iloc[0]
        

        # 혹시 nearest_row 자체에 driver_type이 BLUE로 들어가 있으면 BLUE 처리
        if best_group_driver_type == 'BLUE':
            overlap_df.at[i, "driver_type"]  = driver_type['driver_type']
            overlap_df.at[i, "driver_code"]  = np.nan
            overlap_df.at[i, "cluster_label"]  = cluster_label['cluster_label']
            
            print(f"[INFO] {area} 지역 주문({order_row['shipping_uuid']}) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.")
            continue
        
        # 4) cluster_label 할당
        overlap_df.at[i, "cluster_label"] = cluster_label['cluster_label']
        
        best_group_cluster_label = nearest_row["cluster_label"]

        # 5) 배정된 그룹의 기사 정보 확인
        group_slice = all_KMeans_dfs[
            (all_KMeans_dfs["Area"] == area) & 
            (all_KMeans_dfs["cluster_label"] == best_group_cluster_label)
        ]

        unique_drivers = group_slice[["driver_type", "driver_code", "cluster_label"]].drop_duplicates()
        
        if len(unique_drivers) == 1:
            # 기사정보가 유일하게 한 명이면 그대로 배정
            row_driver = unique_drivers.iloc[0]
            overlap_df.at[i, "driver_type"]  = row_driver["driver_type"]
            overlap_df.at[i, "driver_code"]  = row_driver["driver_code"]
            overlap_df.at[i, "cluster_label"]  = row_driver["cluster_label"]
        else:
            # 기사가 여러 명이거나 아예 없다면 -> 일단 미배정
            overlap_df.at[i, "driver_type"]  = np.nan
            overlap_df.at[i, "driver_code"]  = np.nan
            overlap_df.at[i, "cluster_label"]  = np.nan
        
        print(f"[INFO] {area} 지역 주문({order_row['shipping_uuid']}) → 가장 가까운 데이터의 cluster_label=[{best_group_cluster_label}] 배정 완료")
    
    # 기존 all_KMeans_dfs + overlap_df를 합쳐 최종 반환
    all_KMeans_dfs_plus_overlap = pd.concat([all_KMeans_dfs, overlap_df], ignore_index=True)

    return all_KMeans_dfs_plus_overlap

In [175]:
# 오버랩 클러스터

overlap_df = pd.read_csv('../git_csv/20250318_오버랩데이터.csv')

all_KMeans_dfs_plus_overlap = overlap(overlap_df, all_KMeans_dfs)
today_date = datetime.now().strftime("%Y%m%d")
# 오버랩 최종 시각화
final_map = visualize_clusters(all_KMeans_dfs_plus_overlap)
final_map_filename = f"../우편번호그룹_시각화_html/{today_date}_오버랩포함_우편번호_그룹화_시각화.html"
test_final_map_filename = f"../우편번호그룹_시각화_html/20250318_화요일_오버랩포함_우편번호_그룹_시각화.html"
final_map.save(test_final_map_filename)
print(f"최종 통합 시각화 저장 완료: {test_final_map_filename}")


[INFO] 강남 지역 주문(d645529574df46abbae4ae6bbe7327c1) → 가장 가까운 데이터의 cluster_label=[강남W11] 배정 완료
[INFO] 동안 지역 주문(af67e2200f28405da77a5e9bbe1922d8) → 가장 가까운 데이터의 cluster_label=[동안W1] 배정 완료
[INFO] 성동 지역 주문(37ca89bcb139456484808f8aaeaf0a25) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 김포 지역 주문(6865f8b4274e4f91bb921145d5807fe9) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 하남 지역 주문(bfa85882810e46678d8aaba267fc3344) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 성동 지역 주문(8d6e17a3960846db8f9fbbf3faba483b) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 일산동구 지역 주문(faa0cf4d261a4d2f849869057a828d4c) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 의정부 지역 주문(8ffdae26528b4632aa81c44b50fe8732) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 영등포 지역 주문(2306539b50104e9abc52f3bb8cf5340a) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 광진 지역 주문(2e4720a73b3744a7b0d0d973be0f64b9) → 가장 가까운 데이터의 driver_type이 BLUE, BLUE 로 편입.
[INFO] 중구 지역 주문(23b5be8a8c4543c888362188fa11dce6) → 가장 가까운 데이